# Background Setup

This has nothing to do with the tutorial itself. This is prep work for the Google colab environment.



### Sample Data

Download public clinical data which is a sample of the [medical transcripts public data](https://www.kaggle.com/datasets/tboyle10/medicaltranscriptions/data)

In [1]:
system("curl -o sobp_nlp.csv.gz -L https://www.dropbox.com/scl/fi/garj75z0p7jghp68su82b/sobp_nlp.csv.gz?rlkey=24vuiiz7fluuya1iovy9ajc4e&dl=1")

### Pre Cooked Results

In [2]:
system("curl -o llama.rds -L https://www.dropbox.com/scl/fi/9wcjwi11lr1njmxj4ey55/llama-local_llm_bound.rds?rlkey=gvi20brknyv3j2tdi7cx84zqu&dl=1")
system("curl -o sonnet46.rds -L https://www.dropbox.com/scl/fi/pej6md87ne1w3lt8avqho/sonnet46-remote_llm_bound.rds?rlkey=m3q9gs2k6slitwvsg5ckl361d&dl=1")
system("curl -o gpt5_phq9.rds -L https://www.dropbox.com/scl/fi/4ivokbj83x3pc8lv30fow/gpt5-phq9-remote_llm_bound.rds?rlkey=j3ozao0p722av3n58i4r1kfyh&dl=1")

### Install R libraries

We are not building an LLM, we are using an LLM, and other NLP tools, which can be done from any language (or using CLI tools).

This tutorial will use `R`.

In [3]:
install.packages("httr2")
install.packages("stringi")
install.packages('ellmer')

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘coro’




In [4]:
# Needed for Sql BOW model
system("apt-get install -y sqlite3")

# Needed for ollama install script
system("apt-get install -y zstd", intern = TRUE)

[1] "Reading package lists..."                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      
 [2] "Building dependency tree..."                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   
 [3] "Reading state information..."                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  
 [4] "The following NEW packages will be installed:"                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 
 [5] "  zstd"                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        
 [6] "0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded."                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               
 [7] "Need to get 603 kB of archives."                                                                                       



### Setup MGH CQH Models

The following blocks will define R functions that call into specific scoring code:

*  `score_rdoc_bow` - RDoC 2018 BOW in R
*  `RDOC_SQL_SCHEMA` - RDoC 2018 BOW in Sql
*  `score_rdoc_llm` - RDoC 2024 LLM
*  `score_phq9_llm` - PHQ9 LLM



##### MGH CQH RDoC BOW `score_rdoc_bow`

McCoy TH, Yu S, Hart KL, Castro VM, Brown HE, Rosenquist JN, Doyle AE, Vuijk PJ, Cai T, Perlis RH. High throughput phenotyping for dimensional psychopathology in electronic health records. *Biol Psychiatry*. 2018;83(12):997-1004. DOI: 10.1016/j.biopsych.2018.01.011   PMID: [29496195](https://pubmed.ncbi.nlm.nih.gov/29496195/)

The python 2.7 version of the tool published with the article is on [github](https://github.com/thmccoy/CQH-Dimensional-Phenotyper).



In [ ]:
# @title
score_rdoc_bow <- local({

# CQH Dimensional Phenotyper - R port
#
# Faithful R port of CQHDimensionalPhenotyper_py3.py, the tool described in
# "High Throughput Phenotyping for Dimensional Psychopathology in Electronic
# Health Records" (doi:10.1016/j.biopsych.2018.01.011).
#
# Scores a clinical-text document along five RDoC-style dimensional axes
# (negative, positive, cognitive, social, arousal_regulatory) by counting
# domain-specific tokens and bigrams after a lexical-normalization pass.
#
# This file is sourceable. Public API:
#     score_document(text)              -> named numeric vector (one document)
#     score_documents(texts)            -> data.frame (N documents)
#
# Dependencies: base R + stringi.
# The large data tables (DOMAIN_TOKEN_MAP, TOKEN_LVG_MAP, STOP_WORDS) are
# auto-generated from the Python source via scripts/dump_data_to_r.py and
# live in the # --- BEGIN/END GENERATED DATA --- block below.

if (!requireNamespace("stringi", quietly = TRUE)) {
  stop("CQHDimensionalPhenotyper.R requires the 'stringi' package. ",
       "Install with install.packages('stringi').")
}

# --- BEGIN GENERATED DATA ---
# Auto-generated by scripts/dump_data_to_r.py from
# CQHDimensionalPhenotyper_py3.py. Do not edit by hand.

DOMAIN_TOKEN_MAP <- list(
  negative = c(
    "aggression", "aggressive", "agitated", "anger", "angry",
    "anxiety", "anxiety_disorder", "anxious", "appetite", "ashamed",
    "assaultive", "attack", "avoidance", "avoidant", "blue",
    "calm", "chest", "chest_pain", "chill", "constricted",
    "depressed", "depressed_mood", "depressive", "depressive_disorder", "diaphoretic",
    "diarrhea", "difficulty", "down", "dysphoric", "dysthymic",
    "emotional", "energy", "fatigue", "fearful", "feelings",
    "fever", "frighten", "gain", "guilt", "guilty",
    "head", "helpless", "hopeful", "hopelessness", "hypervigilant",
    "irritable", "lesion", "living", "major", "meaningless",
    "nausea", "nervous", "optimistic", "outburst", "palpitation",
    "panic", "panic_attack", "panic_disorder", "pessimistic", "phobia",
    "phobic", "posttraumatic", "posttraumatic_stress", "refuse", "relaxed",
    "respond", "sad", "scare", "sensory", "sensory_deficit",
    "situation", "startle", "stress", "stress_disorder", "stressor",
    "suicidal", "sweat", "tachycardic", "tearful", "tense",
    "threaten", "trauma", "tremulous", "violence", "violent",
    "weight_gain", "worried", "worth", "worthless"
  ),
  positive = c(
    "abuse", "alcohol", "alcohol_abuse", "alcohol_dependance", "alcohol_withdrawal",
    "appetite", "back", "barbiturate", "blackout", "bowel",
    "compulsive", "count", "couple", "craving", "cut",
    "dependance", "disorder_nos", "drink", "drug", "effect",
    "etoh", "financial", "gamble", "heavy", "heroin",
    "high", "high_school", "hoard", "indication", "induce",
    "induce_mood", "ingestion", "intake", "interest", "intoxicate",
    "intoxication", "liver", "lose", "money", "monitoring",
    "motivate", "motor", "movement", "ordering", "partial",
    "planning", "psychotic_disorder", "related", "remission", "repetitive",
    "report_that", "resource", "routine", "sexual_abuse", "stereotypy",
    "stop", "strategy", "therapy", "with_alcohol", "withdrawal",
    "withdrawal_symptom"
  ),
  cognitive = c(
    "ability", "answer", "aphasic", "attention", "attentive",
    "auditory", "auditory_hallucination", "behavior", "cognitive", "compliance",
    "content", "conversation", "conviction", "current_episode", "decrease_need",
    "delusion", "disinhibit", "disorganisation", "disorganise", "disorganise_behavior",
    "distracted", "distractible", "employ", "engage", "executive",
    "exercise", "factor_include", "fast", "flight", "forget",
    "forgetful", "grandiosity", "hallucinate", "hallucination", "idea",
    "illusion", "impairment", "impulse", "impulsive", "impulsive_behavior",
    "inappropriate", "incoherent", "information", "injurious_behavior", "insertion",
    "intact", "irritable", "memory", "mildly", "name",
    "ongoing", "perception", "planning", "possibly", "problem",
    "psychotic", "psychotic_disorder", "psychotic_feature", "question", "race",
    "recall", "regularly", "related", "schizophrenia", "sensation",
    "sexual_abuse", "sweat", "talk", "term", "testing",
    "thought", "thought_insertion", "type", "ultimately", "understand",
    "understanding", "visual", "visual_hallucination", "voice", "vomiting",
    "with_psychotic", "word"
  ),
  social = c(
    "aggressive", "auditory", "auditory_hallucination", "behavior", "belief",
    "compliance", "concern_that", "content", "control", "current_episode",
    "delusion", "delusional", "disorder_nos", "disorganise", "elicit",
    "empathic", "engaged", "expressive", "fast", "flat",
    "grandiose", "hallucinate", "hallucination", "impulse", "impulse_control",
    "inappropriate", "interview", "isolated", "language", "lonely",
    "misinterpret", "misunderstand", "paranoid", "paranoid_ideation", "partial",
    "prominent", "psychotic", "psychotic_disorder", "psychotic_feature", "regular",
    "related", "theme", "think", "thought_broadcasting", "threaten",
    "type", "vague", "visual", "voice", "wall",
    "with_psychotic", "withdrawn", "worthless"
  ),
  arousal_regulatory = c(
    "abnormality", "agitated", "agitation", "alertness", "alfa",
    "amount", "apnea", "appetite", "architecture", "attention",
    "bipap", "blink", "cardiovascular", "chest_pain", "chill",
    "cigaret", "circadian", "compare", "concentration", "continuity",
    "cortisol", "cpap", "deficit", "deprivation", "difficulty",
    "dizziness", "dream", "duration", "eeg", "emotional",
    "energy", "executive_function", "express", "fatigue", "faze",
    "feelings", "ferritin", "fragmentation", "have_been", "headache",
    "hormone", "hour", "hypersomnia", "identify", "information",
    "insomnia", "irritability", "language", "melatonin", "memory",
    "minutes", "motivate", "motor", "motor_activity", "neck",
    "night", "nightmare", "notice", "offer", "oral",
    "osa", "posttraumatic", "posttraumatic_stress", "psychomotor", "psychomotor_agitation",
    "quantity", "range", "reactivity", "rem_sleep", "respiratory",
    "restless", "restoration", "rhythm", "sensation", "sensitivity",
    "sensory", "sensory_deficit", "sexual_abuse", "sleep_latency", "sleep_study",
    "sleepiness", "someone_else", "spindle", "stability", "stable",
    "startle", "stress", "stress_disorder", "symptom_include", "these_problem",
    "theta", "timing", "tingle", "tired", "trauma",
    "traumatic", "user", "variability", "vigilance", "voluntary",
    "wakefulness", "waking", "weight_gain", "with_general", "worsen"
  )
)

TOKEN_LVG_MAP <- c(
  "abilities" = "ability", "ability" = "ability",
  "abnormalities" = "abnormality", "abnormality" = "abnormality",
  "abuse" = "abuse", "abused" = "abuse",
  "abuses" = "abuse", "abusing" = "abuse",
  "activities" = "activity", "activity" = "activity",
  "aggression" = "aggression", "aggressive" = "aggressive",
  "agitated" = "agitated", "agitation" = "agitation",
  "alcohol" = "alcohol", "alcohols" = "alcohol",
  "alertness" = "alertness", "alfa" = "alfa",
  "alpha" = "alfa", "alphas" = "alfa",
  "amount" = "amount", "amounted" = "amount",
  "amounting" = "amount", "amounts" = "amount",
  "anger" = "anger", "angered" = "anger",
  "angers" = "anger", "angrier" = "angry",
  "angry" = "angry", "answer" = "answer",
  "answered" = "answer", "answering" = "answer",
  "answers" = "answer", "anxieties" = "anxiety",
  "anxiety" = "anxiety", "anxious" = "anxious",
  "aphasic" = "aphasic", "apnea" = "apnea",
  "appetite" = "appetite", "appetites" = "appetite",
  "architecture" = "architecture", "ashamed" = "ashamed",
  "assaultive" = "assaultive", "attack" = "attack",
  "attacked" = "attack", "attacking" = "attack",
  "attacks" = "attack", "attention" = "attention",
  "attentive" = "attentive", "auditory" = "auditory",
  "avoidance" = "avoidance", "avoidant" = "avoidant",
  "back" = "back", "backed" = "back",
  "backs" = "back", "barbiturate" = "barbiturate",
  "barbiturates" = "barbiturate", "behavior" = "behavior",
  "behaviors" = "behavior", "behaviour" = "behavior",
  "behaviours" = "behavior", "belief" = "belief",
  "beliefs" = "belief", "bipap" = "bipap",
  "blackout" = "blackout", "blackouts" = "blackout",
  "blink" = "blink", "blinks" = "blink",
  "blue" = "blue", "bowel" = "bowel",
  "bowels" = "bowel", "broadcasting" = "broadcasting",
  "calm" = "calm", "calmed" = "calm",
  "calmer" = "calm", "calming" = "calm",
  "calms" = "calm", "cardiovascular" = "cardiovascular",
  "chest" = "chest", "chill" = "chill",
  "chilled" = "chill", "chills" = "chill",
  "cigarette" = "cigaret", "cigarettes" = "cigaret",
  "circadian" = "circadian", "cognitive" = "cognitive",
  "compare" = "compare", "compared" = "compare",
  "compares" = "compare", "comparing" = "compare",
  "compliance" = "compliance", "compulsive" = "compulsive",
  "concentration" = "concentration", "concentrations" = "concentration",
  "concern" = "concern", "concerns" = "concern",
  "constricted" = "constricted", "content" = "content",
  "continuity" = "continuity", "control" = "control",
  "controled" = "control", "controling" = "control",
  "controls" = "control", "conversation" = "conversation",
  "conversations" = "conversation", "conviction" = "conviction",
  "convictions" = "conviction", "cortisol" = "cortisol",
  "count" = "count", "counted" = "count",
  "counting" = "count", "counts" = "count",
  "couple" = "couple", "coupled" = "couple",
  "couples" = "couple", "cpap" = "cpap",
  "craving" = "craving", "cravings" = "craving",
  "current" = "current", "cut" = "cut",
  "cuts" = "cut", "decrease" = "decrease",
  "decreased" = "decrease", "decreases" = "decrease",
  "decreasing" = "decrease", "deficit" = "deficit",
  "deficits" = "deficit", "delusion" = "delusion",
  "delusional" = "delusional", "delusions" = "delusion",
  "dependance" = "dependance", "dependence" = "dependance",
  "dependences" = "dependance", "depressed" = "depressed",
  "depressive" = "depressive", "deprivation" = "deprivation",
  "diaphoretic" = "diaphoretic", "diarrhea" = "diarrhea",
  "difficulties" = "difficulty", "difficulty" = "difficulty",
  "disinhibit" = "disinhibit", "disinhibited" = "disinhibit",
  "disorder" = "disorder", "disorders" = "disorder",
  "disorganised" = "disorganise", "disorganization" = "disorganisation",
  "disorganize" = "disorganise", "disorganized" = "disorganise",
  "distracted" = "distracted", "distractible" = "distractible",
  "dizziness" = "dizziness", "dizzyness" = "dizziness",
  "downing" = "down", "downs" = "down",
  "drank" = "drink", "dream" = "dream",
  "dreamed" = "dream", "dreaming" = "dream",
  "dreams" = "dream", "drink" = "drink",
  "drinking" = "drink", "drinks" = "drink",
  "drug" = "drug", "drugging" = "drug",
  "drugs" = "drug", "duration" = "duration",
  "dysphoric" = "dysphoric", "dysthymic" = "dysthymic",
  "eeg" = "eeg", "eegs" = "eeg",
  "effect" = "effect", "effected" = "effect",
  "effecting" = "effect", "effects" = "effect",
  "elicit" = "elicit", "elicited" = "elicit",
  "eliciting" = "elicit", "elicits" = "elicit",
  "emotional" = "emotional", "empathic" = "empathic",
  "employ" = "employ", "employed" = "employ",
  "employing" = "employ", "employs" = "employ",
  "energies" = "energy", "energy" = "energy",
  "engage" = "engage", "engaged" = "engaged",
  "engages" = "engage", "engaging" = "engage",
  "episode" = "episode", "episodes" = "episode",
  "etoh" = "etoh", "executive" = "executive",
  "exercise" = "exercise", "exercised" = "exercise",
  "exercises" = "exercise", "exercising" = "exercise",
  "exercizes" = "exercise", "express" = "express",
  "expressed" = "express", "expresses" = "express",
  "expressing" = "express", "expressive" = "expressive",
  "factor" = "factor", "factored" = "factor",
  "factors" = "factor", "fast" = "fast",
  "fasted" = "fast", "faster" = "fast",
  "fastest" = "fast", "fasting" = "fast",
  "fatigue" = "fatigue", "fatigued" = "fatigue",
  "fearful" = "fearful", "feature" = "feature",
  "featured" = "feature", "features" = "feature",
  "featuring" = "feature", "feelings" = "feelings",
  "ferritin" = "ferritin", "fever" = "fever",
  "fevers" = "fever", "financial" = "financial",
  "flat" = "flat", "flight" = "flight",
  "flights" = "flight", "forget" = "forget",
  "forgetful" = "forgetful", "forgets" = "forget",
  "forgetting" = "forget", "forgot" = "forget",
  "forgotten" = "forget", "fragmentation" = "fragmentation",
  "frighten" = "frighten", "frightened" = "frighten",
  "frightens" = "frighten", "function" = "function",
  "functioned" = "function", "functioning" = "function",
  "functions" = "function", "gain" = "gain",
  "gained" = "gain", "gaining" = "gain",
  "gamble" = "gamble", "gambled" = "gamble",
  "gambles" = "gamble", "gambling" = "gamble",
  "general" = "general", "grandiose" = "grandiose",
  "grandiosity" = "grandiosity", "guilt" = "guilt",
  "guilty" = "guilty", "hallucinate" = "hallucinate",
  "hallucinated" = "hallucinate", "hallucinating" = "hallucinate",
  "hallucination" = "hallucination", "hallucinations" = "hallucination",
  "head" = "head", "headache" = "headache",
  "headaches" = "headache", "heads" = "head",
  "heavier" = "heavy", "heaviest" = "heavy",
  "heavy" = "heavy", "helpless" = "helpless",
  "heroin" = "heroin", "high" = "high",
  "higher" = "high", "highest" = "high",
  "highs" = "high", "hoard" = "hoard",
  "hoarded" = "hoard", "hoarding" = "hoard",
  "hoards" = "hoard", "hopeful" = "hopeful",
  "hopelessness" = "hopelessness", "hormone" = "hormone",
  "hormones" = "hormone", "hour" = "hour",
  "hours" = "hour", "hypersomnia" = "hypersomnia",
  "hypervigilant" = "hypervigilant", "idea" = "idea",
  "ideas" = "idea", "ideation" = "ideation",
  "ideations" = "ideation", "identified" = "identify",
  "identifies" = "identify", "identify" = "identify",
  "identifying" = "identify", "illusion" = "illusion",
  "illusions" = "illusion", "impairment" = "impairment",
  "impairments" = "impairment", "impulse" = "impulse",
  "impulses" = "impulse", "impulsive" = "impulsive",
  "inappropriate" = "inappropriate", "include" = "include",
  "includes" = "include", "incoherent" = "incoherent",
  "indication" = "indication", "indications" = "indication",
  "induce" = "induce", "induced" = "induce",
  "induces" = "induce", "inducing" = "induce",
  "information" = "information", "ingestion" = "ingestion",
  "ingestions" = "ingestion", "injurious" = "injurious",
  "insertion" = "insertion", "insertions" = "insertion",
  "insomnia" = "insomnia", "intact" = "intact",
  "intake" = "intake", "intakes" = "intake",
  "interest" = "interest", "interests" = "interest",
  "interview" = "interview", "interviewed" = "interview",
  "interviewing" = "interview", "interviews" = "interview",
  "intoxicated" = "intoxicate", "intoxication" = "intoxication",
  "irritability" = "irritability", "irritable" = "irritable",
  "isolated" = "isolated", "language" = "language",
  "languaged" = "language", "languages" = "language",
  "latencies" = "latency", "latency" = "latency",
  "lesion" = "lesion", "lesioning" = "lesion",
  "lesions" = "lesion", "liver" = "liver",
  "living" = "living", "lonelier" = "lonely",
  "lonely" = "lonely", "lose" = "lose",
  "loses" = "lose", "losing" = "lose",
  "lost" = "lose", "major" = "major",
  "majored" = "major", "majoring" = "major",
  "majors" = "major", "meaningless" = "meaningless",
  "melatonin" = "melatonin", "memories" = "memory",
  "memory" = "memory", "mildly" = "mildly",
  "minutes" = "minutes", "misinterpret" = "misinterpret",
  "misinterpreted" = "misinterpret", "misinterpreting" = "misinterpret",
  "misinterprets" = "misinterpret", "misunderstand" = "misunderstand",
  "misunderstands" = "misunderstand", "misunderstood" = "misunderstand",
  "money" = "money", "monitoring" = "monitoring",
  "mood" = "mood", "moods" = "mood",
  "motivate" = "motivate", "motivated" = "motivate",
  "motivating" = "motivate", "motor" = "motor",
  "movement" = "movement", "movements" = "movement",
  "named" = "name", "names" = "name",
  "naming" = "name", "nausea" = "nausea",
  "nauseas" = "nausea", "neck" = "neck",
  "necks" = "neck", "needed" = "need",
  "needing" = "need", "nervous" = "nervous",
  "night" = "night", "nightmare" = "nightmare",
  "nightmares" = "nightmare", "nights" = "night",
  "notice" = "notice", "noticed" = "notice",
  "notices" = "notice", "noticing" = "notice",
  "offer" = "offer", "offered" = "offer",
  "offers" = "offer", "ongoing" = "ongoing",
  "optimistic" = "optimistic", "oral" = "oral",
  "ordering" = "ordering", "osa" = "osa",
  "outburst" = "outburst", "outbursts" = "outburst",
  "pain" = "pain", "pained" = "pain",
  "pains" = "pain", "palpitation" = "palpitation",
  "palpitations" = "palpitation", "panic" = "panic",
  "panicked" = "panic", "panicking" = "panic",
  "panics" = "panic", "paranoid" = "paranoid",
  "paranoids" = "paranoid", "partial" = "partial",
  "perception" = "perception", "perceptions" = "perception",
  "pessimistic" = "pessimistic", "phase" = "faze",
  "phases" = "faze", "phasing" = "faze",
  "phobia" = "phobia", "phobias" = "phobia",
  "phobic" = "phobic", "planning" = "planning",
  "possibly" = "possibly", "posttraumatic" = "posttraumatic",
  "problem" = "problem", "problems" = "problem",
  "prominent" = "prominent", "psychomotor" = "psychomotor",
  "psychotic" = "psychotic", "psychotics" = "psychotic",
  "quantities" = "quantity", "quantity" = "quantity",
  "question" = "question", "questioned" = "question",
  "questions" = "question", "race" = "race",
  "raced" = "race", "races" = "race",
  "racing" = "race", "range" = "range",
  "ranged" = "range", "ranges" = "range",
  "ranging" = "range", "reactivity" = "reactivity",
  "recall" = "recall", "recalled" = "recall",
  "recalling" = "recall", "recalls" = "recall",
  "refuse" = "refuse", "refused" = "refuse",
  "refuses" = "refuse", "refusing" = "refuse",
  "regular" = "regular", "regularly" = "regularly",
  "related" = "related", "relaxed" = "relaxed",
  "rem" = "rem", "remission" = "remission",
  "remissions" = "remission", "repetitive" = "repetitive",
  "report" = "report", "reported" = "report",
  "reports" = "report", "resource" = "resource",
  "resources" = "resource", "respiratory" = "respiratory",
  "respond" = "respond", "responded" = "respond",
  "responding" = "respond", "responds" = "respond",
  "restless" = "restless", "restoration" = "restoration",
  "rhythm" = "rhythm", "rhythms" = "rhythm",
  "routine" = "routine", "routines" = "routine",
  "sad" = "sad", "scare" = "scare",
  "scared" = "scare", "scares" = "scare",
  "scaring" = "scare", "schizophrenia" = "schizophrenia",
  "school" = "school", "schooled" = "school",
  "schooling" = "school", "schools" = "school",
  "sensation" = "sensation", "sensations" = "sensation",
  "sensitivities" = "sensitivity", "sensitivity" = "sensitivity",
  "sensory" = "sensory", "sexual" = "sexual",
  "situation" = "situation", "situations" = "situation",
  "sleep" = "sleep", "sleepiness" = "sleepiness",
  "sleeping" = "sleep", "sleeps" = "sleep",
  "slept" = "sleep", "spindle" = "spindle",
  "spindles" = "spindle", "stability" = "stability",
  "stable" = "stable", "stables" = "stable",
  "startle" = "startle", "startled" = "startle",
  "startles" = "startle", "stereotypies" = "stereotypy",
  "stereotypy" = "stereotypy", "stop" = "stop",
  "stopped" = "stop", "stopping" = "stop",
  "stops" = "stop", "strategies" = "strategy",
  "strategy" = "strategy", "stress" = "stress",
  "stressed" = "stress", "stresses" = "stress",
  "stressing" = "stress", "stressor" = "stressor",
  "stressors" = "stressor", "studies" = "study",
  "study" = "study", "studying" = "study",
  "suicidal" = "suicidal", "sweat" = "sweat",
  "sweated" = "sweat", "sweating" = "sweat",
  "sweats" = "sweat", "symptom" = "symptom",
  "symptoms" = "symptom", "tachycardic" = "tachycardic",
  "talk" = "talk", "talked" = "talk",
  "talking" = "talk", "talks" = "talk",
  "tearful" = "tearful", "tense" = "tense",
  "tensing" = "tense", "term" = "term",
  "termed" = "term", "terms" = "term",
  "testing" = "testing", "testings" = "testing",
  "theme" = "theme", "themes" = "theme",
  "therapies" = "therapy", "therapy" = "therapy",
  "theta" = "theta", "thinking" = "think",
  "thinks" = "think", "thought" = "thought",
  "thoughts" = "thought", "threaten" = "threaten",
  "threatened" = "threaten", "threatening" = "threaten",
  "threatens" = "threaten", "timing" = "timing",
  "timings" = "timing", "tingle" = "tingle",
  "tingling" = "tingle", "tired" = "tired",
  "trauma" = "trauma", "traumas" = "trauma",
  "traumata" = "trauma", "traumatic" = "traumatic",
  "tremulous" = "tremulous", "type" = "type",
  "types" = "type", "ultimately" = "ultimately",
  "understand" = "understand", "understanding" = "understanding",
  "understandings" = "understanding", "understands" = "understand",
  "understood" = "understand", "user" = "user",
  "users" = "user", "vague" = "vague",
  "variability" = "variability", "vigilance" = "vigilance",
  "violence" = "violence", "violent" = "violent",
  "visual" = "visual", "voice" = "voice",
  "voices" = "voice", "voicing" = "voice",
  "voluntary" = "voluntary", "vomiting" = "vomiting",
  "wakefulness" = "wakefulness", "waking" = "waking",
  "wakings" = "waking", "wall" = "wall",
  "walls" = "wall", "weight" = "weight",
  "weighted" = "weight", "weights" = "weight",
  "withdrawal" = "withdrawal", "withdrawals" = "withdrawal",
  "withdrawn" = "withdrawn", "word" = "word",
  "worded" = "word", "wording" = "word",
  "words" = "word", "worried" = "worried",
  "worsen" = "worsen", "worsened" = "worsen",
  "worsening" = "worsen", "worsens" = "worsen",
  "worth" = "worth", "worthless" = "worthless"
)

STOP_WORDS <- c(
  "a's", "able", "about", "above", "according",
  "accordingly", "across", "actually", "after", "afterwards",
  "again", "against", "ain't", "all", "allow",
  "allows", "almost", "alone", "along", "already",
  "also", "although", "always", "am", "among",
  "amongst", "an", "and", "another", "any",
  "anybody", "anyhow", "anyone", "anything", "anyway",
  "anyways", "anywhere", "apart", "appear", "appreciate",
  "appropriate", "are", "aren't", "around", "as",
  "aside", "ask", "asking", "associated", "at",
  "available", "away", "awfully", "be", "became",
  "because", "become", "becomes", "becoming", "before",
  "beforehand", "behind", "being", "believe", "below",
  "beside", "besides", "best", "better", "between",
  "beyond", "both", "brief", "but", "by",
  "c'mon", "c's", "came", "can", "can't",
  "cannot", "cant", "cause", "causes", "certain",
  "certainly", "changes", "clearly", "co", "com",
  "come", "comes", "concerning", "consequently", "consider",
  "considering", "contain", "containing", "contains", "corresponding",
  "could", "couldn't", "course", "currently", "definitely",
  "described", "despite", "did", "didn't", "different",
  "do", "does", "doesn't", "doing", "don't",
  "done", "down", "downwards", "during", "each",
  "edu", "eg", "eight", "either", "elsewhere",
  "enough", "entirely", "especially", "et", "etc",
  "even", "ever", "every", "everybody", "everyone",
  "everything", "everywhere", "ex", "exactly", "example",
  "except", "far", "few", "fifth", "first",
  "five", "followed", "following", "follows", "for",
  "former", "formerly", "forth", "four", "from",
  "further", "furthermore", "get", "gets", "getting",
  "given", "gives", "go", "goes", "going",
  "gone", "got", "gotten", "greetings", "had",
  "hadn't", "happens", "hardly", "has", "hasn't",
  "haven't", "having", "he", "he's", "hello",
  "help", "hence", "her", "here", "here's",
  "hereafter", "hereby", "herein", "hereupon", "hers",
  "herself", "hi", "him", "himself", "his",
  "hither", "hopefully", "how", "howbeit", "however",
  "i'd", "i'll", "i'm", "i've", "ie",
  "if", "ignored", "immediate", "in", "inasmuch",
  "inc", "indeed", "indicate", "indicated", "indicates",
  "inner", "insofar", "instead", "into", "inward",
  "is", "isn't", "it", "it'd", "it'll",
  "it's", "its", "itself", "just", "keep",
  "keeps", "kept", "know", "knows", "known",
  "last", "lately", "later", "latter", "latterly",
  "least", "less", "lest", "let", "let's",
  "like", "liked", "likely", "little", "look",
  "looking", "looks", "ltd", "mainly", "many",
  "may", "maybe", "me", "mean", "meanwhile",
  "merely", "might", "more", "moreover", "most",
  "mostly", "much", "must", "my", "myself",
  "name", "namely", "nd", "near", "nearly",
  "necessary", "needs", "neither", "never", "nevertheless",
  "new", "next", "nine", "no", "nobody",
  "non", "none", "noone", "nor", "normally",
  "not", "nothing", "novel", "now", "nowhere",
  "obviously", "of", "off", "often", "oh",
  "ok", "okay", "old", "on", "once",
  "one", "ones", "only", "onto", "or",
  "other", "others", "otherwise", "ought", "our",
  "ours", "ourselves", "out", "outside", "over",
  "overall", "own", "particular", "particularly", "per",
  "perhaps", "placed", "please", "plus", "possible",
  "presumably", "probably", "provides", "que", "quite",
  "qv", "rather", "rd", "re", "really",
  "reasonably", "regarding", "regardless", "regards", "relatively",
  "respectively", "right", "said", "same", "saw",
  "say", "saying", "says", "second", "secondly",
  "see", "seeing", "seem", "seemed", "seeming",
  "seems", "seen", "self", "selves", "sensible",
  "sent", "serious", "seriously", "seven", "several",
  "shall", "she", "should", "shouldn't", "since",
  "six", "so", "some", "somebody", "somehow",
  "something", "sometime", "sometimes", "somewhat", "somewhere",
  "soon", "sorry", "specified", "specify", "specifying",
  "still", "sub", "such", "sup", "sure",
  "t's", "take", "taken", "tell", "tends",
  "th", "than", "thank", "thanks", "thanx",
  "that's", "thats", "the", "their", "theirs",
  "them", "themselves", "then", "thence", "there",
  "there's", "thereafter", "thereby", "therefore", "therein",
  "theres", "thereupon", "they", "they'd", "they'll",
  "they're", "they've", "think", "third", "this",
  "thorough", "thoroughly", "those", "though", "three",
  "through", "throughout", "thru", "thus", "to",
  "together", "too", "took", "toward", "towards",
  "tried", "tries", "truly", "try", "trying",
  "twice", "two", "un", "under", "unfortunately",
  "unless", "unlikely", "until", "unto", "up",
  "upon", "us", "use", "used", "useful",
  "uses", "using", "usually", "value", "various",
  "very", "via", "viz", "vs", "want",
  "wants", "was", "wasn't", "way", "we",
  "we'd", "we'll", "we're", "we've", "welcome",
  "well", "went", "were", "weren't", "what",
  "what's", "whatever", "when", "whence", "whenever",
  "where", "where's", "whereafter", "whereas", "whereby",
  "wherein", "whereupon", "wherever", "whether", "which",
  "while", "whither", "who", "who's", "whoever",
  "whole", "whom", "whose", "why", "will",
  "willing", "wish", "within", "without", "won't",
  "wonder", "would", "would", "wouldn't", "yes",
  "yet", "you", "you'd", "you'll", "you're",
  "you've", "your", "yours", "yourself", "yourselves",
  "zero"
)

# --- END GENERATED DATA ---

# Sorted, unique union of all domain tokens. Used as the universe of countable
# terms (both unigrams and underscore-joined bigrams).
TOKENS <- sort(unique(unlist(DOMAIN_TOKEN_MAP, use.names = FALSE)))

# Public-facing column order, mirroring the CSV header in the Python tool.
# Edit this if you add a new dimensional axis to DOMAIN_TOKEN_MAP.
CQH_OUTPUT_COLUMNS <- c(
  "length", "match_count",
  "negative", "positive", "cognitive", "social", "arousal_regulatory",
  "elapsed_time"
)


# ---- internal helpers ------------------------------------------------------

clean_text <- function(raw_text) {
  # Match Python: raw_text.encode("ascii", "ignore").decode("ascii")
  # which *drops* (rather than transliterates) non-ASCII bytes. We route
  # through enc2utf8() so the encoding is explicit and the same regardless of
  # the user's locale, then iconv from UTF-8 to ASCII to drop the rest.
  if (is.na(raw_text)) return("")
  text <- iconv(enc2utf8(raw_text), from = "UTF-8", to = "ASCII", sub = "")
  text <- stringi::stri_trans_tolower(text)
  text <- stringi::stri_replace_all_regex(text, "[^A-Za-z0-9]+", " ")
  # The (?m)^\s+ pass is a faithful copy of the Python WHITE_SPACE regex.
  # After the previous step strips newlines too, it only ever trims leading
  # whitespace at the start of the (now single-line) string -- but we keep it
  # for byte-parity with the reference implementation.
  stringi::stri_replace_all_regex(text, "(?m)^\\s+", " ")
}

tokenize <- function(text, stopwords, lookup) {
  # Python: text.split() drops empty tokens, .isdigit() filters pure digit
  # tokens, len(tk) > 2 filters short tokens, then optional LVG normalization.
  tk <- stringi::stri_split_regex(text, "\\s+", omit_empty = TRUE)[[1]]
  if (length(tk) == 0L) return(character(0))
  tk <- tk[!grepl("^[0-9]+$", tk) & nchar(tk) > 2L & !(tk %in% stopwords)]
  if (length(tk) == 0L) return(character(0))
  idx <- match(tk, names(lookup))
  hit <- !is.na(idx)
  tk[hit] <- unname(lookup[idx[hit]])
  tk
}

as_bigrams <- function(tokens) {
  # Mirror Python's `zip(tokens, tokens[1:] + [''])`. The trailing empty
  # string is critical: it preserves length(bigrams) == length(tokens) and
  # ensures every token appears in the `a` slot of exactly one pair so unigram
  # counts are not under-reported by one.
  if (length(tokens) == 0L) return(list(a = character(0), b = character(0)))
  list(a = tokens, b = c(tokens[-1L], ""))
}

count_terms <- function(bigrams, to_count) {
  # Two simultaneous count passes per pair (a, b):
  #   1. unigram pass: increment to_count[a] if present
  #   2. bigram pass:  increment to_count["a_b"] if present
  # Both can fire on the same iteration; that's intentional.
  #
  # tabulate() ignores NAs and out-of-range indices, so we don't have to
  # filter the match() results manually -- and the empty case (zero bigrams)
  # falls out for free as tabulate(integer(0), nbins=N) -> N zeros.
  uni_idx <- match(bigrams$a, to_count)
  bi_idx  <- match(paste(bigrams$a, bigrams$b, sep = "_"), to_count)
  counts  <- setNames(
    tabulate(c(uni_idx, bi_idx), nbins = length(to_count)),
    to_count
  )
  attr(counts, "len") <- length(bigrams$a)
  counts
}

domain_rollup <- function(counts, domain_map = DOMAIN_TOKEN_MAP) {
  lapply(domain_map, function(toks) counts[toks])
}

domain_density <- function(by_domain) {
  vapply(by_domain, function(v) round(mean(v > 0), 5), numeric(1))
}


# ---- public API ------------------------------------------------------------

#' Score a single clinical document along the five CQH dimensional axes.
#'
#' @param text A length-1 character vector containing the document.
#' @return Named numeric vector with elements named by CQH_OUTPUT_COLUMNS.
score_document <- function(text,
                           stopwords  = STOP_WORDS,
                           to_count   = TOKENS,
                           lookup     = TOKEN_LVG_MAP,
                           domain_map = DOMAIN_TOKEN_MAP) {
  if (!is.character(text) || length(text) != 1L) {
    stop("score_document() expects a length-1 character vector; ",
         "use score_documents() for multiple documents.")
  }
  t0 <- Sys.time()
  cleaned <- clean_text(text)
  tokens  <- tokenize(cleaned, stopwords, lookup)
  bigrams <- as_bigrams(tokens)
  counts  <- count_terms(bigrams, to_count)
  dens    <- domain_density(domain_rollup(counts, domain_map))
  elapsed <- as.numeric(difftime(Sys.time(), t0, units = "secs"))

  # Python's reduce_counts() does sum(counts.values()) - counts['len'] because
  # 'len' is stored *inside* the counts dict. Here 'len' is an attribute, so
  # sum(counts) already excludes it -- no subtraction needed.
  vals <- c(
    length       = attr(counts, "len"),
    match_count  = sum(counts),
    dens,
    elapsed_time = elapsed
  )
  vals[CQH_OUTPUT_COLUMNS]
}

#' Score many clinical documents at once.
#'
#' @param texts A character vector (one element per document).
#' @return A data.frame with one row per input document and columns named
#'   by CQH_OUTPUT_COLUMNS, in that order.
score_documents <- function(texts,
                            stopwords  = STOP_WORDS,
                            to_count   = TOKENS,
                            lookup     = TOKEN_LVG_MAP,
                            domain_map = DOMAIN_TOKEN_MAP) {
  if (!is.character(texts)) {
    stop("score_documents() expects a character vector.")
  }
  if (length(texts) == 0L) {
    return(as.data.frame(
      matrix(numeric(0), nrow = 0, ncol = length(CQH_OUTPUT_COLUMNS),
             dimnames = list(NULL, CQH_OUTPUT_COLUMNS))
    ))
  }
  rows <- lapply(texts, score_document,
                 stopwords  = stopwords,
                 to_count   = to_count,
                 lookup     = lookup,
                 domain_map = domain_map)
  as.data.frame(do.call(rbind, rows))
}

score_documents

})

##### MGH CQH RDoC SQL

This is the ~ the same scoring as `score_rdoc_bow` but done in Sqlite instead of R / Python

In [6]:
# @title
RDOC_SQL_SCHEMA <- r"---(
BEGIN;

DROP VIEW IF EXISTS cqh_note_scores;
DROP VIEW IF EXISTS cqh_note_domain_hits;
DROP VIEW IF EXISTS cqh_note_term_counts;
DROP VIEW IF EXISTS cqh_note_tokens;
DROP VIEW IF EXISTS cqh_note_tokens_raw;
DROP VIEW IF EXISTS cqh_notes_cleaned;
DROP VIEW IF EXISTS cqh_notes_source;

DROP TABLE IF EXISTS cqh_all_terms;
DROP TABLE IF EXISTS cqh_domain_terms;
DROP TABLE IF EXISTS cqh_normalize;
DROP TABLE IF EXISTS cqh_stopwords;

CREATE TABLE IF NOT EXISTS notes (
  id INTEGER PRIMARY KEY,
  note_text TEXT
);

CREATE TABLE cqh_stopwords (
  word TEXT PRIMARY KEY
);

CREATE TABLE cqh_normalize (
  raw TEXT PRIMARY KEY,
  canon TEXT NOT NULL
);

CREATE TABLE cqh_domain_terms (
  domain TEXT NOT NULL,
  term TEXT NOT NULL,
  PRIMARY KEY (domain, term)
);

CREATE INDEX cqh_domain_terms_term_idx ON cqh_domain_terms(term);

CREATE TABLE cqh_all_terms (
  term TEXT PRIMARY KEY
);

INSERT INTO cqh_stopwords (word) VALUES
('a''s'),
('able'),
('about'),
('above'),
('according'),
('accordingly'),
('across'),
('actually'),
('after'),
('afterwards'),
('again'),
('against'),
('ain''t'),
('all'),
('allow'),
('allows'),
('almost'),
('alone'),
('along'),
('already'),
('also'),
('although'),
('always'),
('am'),
('among'),
('amongst'),
('an'),
('and'),
('another'),
('any'),
('anybody'),
('anyhow'),
('anyone'),
('anything'),
('anyway'),
('anyways'),
('anywhere'),
('apart'),
('appear'),
('appreciate'),
('appropriate'),
('are'),
('aren''t'),
('around'),
('as'),
('aside'),
('ask'),
('asking'),
('associated'),
('at'),
('available'),
('away'),
('awfully'),
('be'),
('became'),
('because'),
('become'),
('becomes'),
('becoming'),
('before'),
('beforehand'),
('behind'),
('being'),
('believe'),
('below'),
('beside'),
('besides'),
('best'),
('better'),
('between'),
('beyond'),
('both'),
('brief'),
('but'),
('by'),
('c''mon'),
('c''s'),
('came'),
('can'),
('can''t'),
('cannot'),
('cant'),
('cause'),
('causes'),
('certain'),
('certainly'),
('changes'),
('clearly'),
('co'),
('com'),
('come'),
('comes'),
('concerning'),
('consequently'),
('consider'),
('considering'),
('contain'),
('containing'),
('contains'),
('corresponding'),
('could'),
('couldn''t'),
('course'),
('currently'),
('definitely'),
('described'),
('despite'),
('did'),
('didn''t'),
('different'),
('do'),
('does'),
('doesn''t'),
('doing'),
('don''t'),
('done'),
('down'),
('downwards'),
('during'),
('each'),
('edu'),
('eg'),
('eight'),
('either'),
('elsewhere'),
('enough'),
('entirely'),
('especially'),
('et'),
('etc'),
('even'),
('ever'),
('every'),
('everybody'),
('everyone'),
('everything'),
('everywhere'),
('ex'),
('exactly'),
('example'),
('except'),
('far'),
('few'),
('fifth'),
('first'),
('five'),
('followed'),
('following'),
('follows'),
('for'),
('former'),
('formerly'),
('forth'),
('four'),
('from'),
('further'),
('furthermore'),
('get'),
('gets'),
('getting'),
('given'),
('gives'),
('go'),
('goes'),
('going'),
('gone'),
('got'),
('gotten'),
('greetings'),
('had'),
('hadn''t'),
('happens'),
('hardly'),
('has'),
('hasn''t'),
('haven''t'),
('having'),
('he'),
('he''s'),
('hello'),
('help'),
('hence'),
('her'),
('here'),
('here''s'),
('hereafter'),
('hereby'),
('herein'),
('hereupon'),
('hers'),
('herself'),
('hi'),
('him'),
('himself'),
('his'),
('hither'),
('hopefully'),
('how'),
('howbeit'),
('however'),
('i''d'),
('i''ll'),
('i''m'),
('i''ve'),
('ie'),
('if'),
('ignored'),
('immediate'),
('in'),
('inasmuch'),
('inc'),
('indeed'),
('indicate'),
('indicated'),
('indicates'),
('inner'),
('insofar'),
('instead'),
('into'),
('inward'),
('is'),
('isn''t'),
('it'),
('it''d'),
('it''ll'),
('it''s'),
('its'),
('itself'),
('just'),
('keep'),
('keeps'),
('kept'),
('know'),
('known'),
('knows'),
('last'),
('lately'),
('later'),
('latter'),
('latterly'),
('least'),
('less'),
('lest'),
('let'),
('let''s'),
('like'),
('liked'),
('likely'),
('little'),
('look');

INSERT INTO cqh_stopwords (word) VALUES
('looking'),
('looks'),
('ltd'),
('mainly'),
('many'),
('may'),
('maybe'),
('me'),
('mean'),
('meanwhile'),
('merely'),
('might'),
('more'),
('moreover'),
('most'),
('mostly'),
('much'),
('must'),
('my'),
('myself'),
('name'),
('namely'),
('nd'),
('near'),
('nearly'),
('necessary'),
('needs'),
('neither'),
('never'),
('nevertheless'),
('new'),
('next'),
('nine'),
('no'),
('nobody'),
('non'),
('none'),
('noone'),
('nor'),
('normally'),
('not'),
('nothing'),
('novel'),
('now'),
('nowhere'),
('obviously'),
('of'),
('off'),
('often'),
('oh'),
('ok'),
('okay'),
('old'),
('on'),
('once'),
('one'),
('ones'),
('only'),
('onto'),
('or'),
('other'),
('others'),
('otherwise'),
('ought'),
('our'),
('ours'),
('ourselves'),
('out'),
('outside'),
('over'),
('overall'),
('own'),
('particular'),
('particularly'),
('per'),
('perhaps'),
('placed'),
('please'),
('plus'),
('possible'),
('presumably'),
('probably'),
('provides'),
('que'),
('quite'),
('qv'),
('rather'),
('rd'),
('re'),
('really'),
('reasonably'),
('regarding'),
('regardless'),
('regards'),
('relatively'),
('respectively'),
('right'),
('said'),
('same'),
('saw'),
('say'),
('saying'),
('says'),
('second'),
('secondly'),
('see'),
('seeing'),
('seem'),
('seemed'),
('seeming'),
('seems'),
('seen'),
('self'),
('selves'),
('sensible'),
('sent'),
('serious'),
('seriously'),
('seven'),
('several'),
('shall'),
('she'),
('should'),
('shouldn''t'),
('since'),
('six'),
('so'),
('some'),
('somebody'),
('somehow'),
('something'),
('sometime'),
('sometimes'),
('somewhat'),
('somewhere'),
('soon'),
('sorry'),
('specified'),
('specify'),
('specifying'),
('still'),
('sub'),
('such'),
('sup'),
('sure'),
('t''s'),
('take'),
('taken'),
('tell'),
('tends'),
('th'),
('than'),
('thank'),
('thanks'),
('thanx'),
('that''s'),
('thats'),
('the'),
('their'),
('theirs'),
('them'),
('themselves'),
('then'),
('thence'),
('there'),
('there''s'),
('thereafter'),
('thereby'),
('therefore'),
('therein'),
('theres'),
('thereupon'),
('they'),
('they''d'),
('they''ll'),
('they''re'),
('they''ve'),
('think'),
('third'),
('this'),
('thorough'),
('thoroughly'),
('those'),
('though'),
('three'),
('through'),
('throughout'),
('thru'),
('thus'),
('to'),
('together'),
('too'),
('took'),
('toward'),
('towards'),
('tried'),
('tries'),
('truly'),
('try'),
('trying'),
('twice'),
('two'),
('un'),
('under'),
('unfortunately'),
('unless'),
('unlikely'),
('until'),
('unto'),
('up'),
('upon'),
('us'),
('use'),
('used'),
('useful'),
('uses'),
('using'),
('usually'),
('value'),
('various'),
('very'),
('via'),
('viz'),
('vs'),
('want'),
('wants'),
('was'),
('wasn''t'),
('way'),
('we'),
('we''d'),
('we''ll'),
('we''re'),
('we''ve'),
('welcome'),
('well'),
('went'),
('were'),
('weren''t'),
('what'),
('what''s'),
('whatever'),
('when'),
('whence'),
('whenever'),
('where'),
('where''s'),
('whereafter'),
('whereas'),
('whereby');

INSERT INTO cqh_stopwords (word) VALUES
('wherein'),
('whereupon'),
('wherever'),
('whether'),
('which'),
('while'),
('whither'),
('who'),
('who''s'),
('whoever'),
('whole'),
('whom'),
('whose'),
('why'),
('will'),
('willing'),
('wish'),
('within'),
('without'),
('won''t'),
('wonder'),
('would'),
('wouldn''t'),
('yes'),
('yet'),
('you'),
('you''d'),
('you''ll'),
('you''re'),
('you''ve'),
('your'),
('yours'),
('yourself'),
('yourselves'),
('zero');

INSERT INTO cqh_normalize (raw, canon) VALUES
('abilities', 'ability'),
('ability', 'ability'),
('abnormalities', 'abnormality'),
('abnormality', 'abnormality'),
('abuse', 'abuse'),
('abused', 'abuse'),
('abuses', 'abuse'),
('abusing', 'abuse'),
('activities', 'activity'),
('activity', 'activity'),
('aggression', 'aggression'),
('aggressive', 'aggressive'),
('agitated', 'agitated'),
('agitation', 'agitation'),
('alcohol', 'alcohol'),
('alcohols', 'alcohol'),
('alertness', 'alertness'),
('alfa', 'alfa'),
('alpha', 'alfa'),
('alphas', 'alfa'),
('amount', 'amount'),
('amounted', 'amount'),
('amounting', 'amount'),
('amounts', 'amount'),
('anger', 'anger'),
('angered', 'anger'),
('angers', 'anger'),
('angrier', 'angry'),
('angry', 'angry'),
('answer', 'answer'),
('answered', 'answer'),
('answering', 'answer'),
('answers', 'answer'),
('anxieties', 'anxiety'),
('anxiety', 'anxiety'),
('anxious', 'anxious'),
('aphasic', 'aphasic'),
('apnea', 'apnea'),
('appetite', 'appetite'),
('appetites', 'appetite'),
('architecture', 'architecture'),
('ashamed', 'ashamed'),
('assaultive', 'assaultive'),
('attack', 'attack'),
('attacked', 'attack'),
('attacking', 'attack'),
('attacks', 'attack'),
('attention', 'attention'),
('attentive', 'attentive'),
('auditory', 'auditory'),
('avoidance', 'avoidance'),
('avoidant', 'avoidant'),
('back', 'back'),
('backed', 'back'),
('backs', 'back'),
('barbiturate', 'barbiturate'),
('barbiturates', 'barbiturate'),
('behavior', 'behavior'),
('behaviors', 'behavior'),
('behaviour', 'behavior'),
('behaviours', 'behavior'),
('belief', 'belief'),
('beliefs', 'belief'),
('bipap', 'bipap'),
('blackout', 'blackout'),
('blackouts', 'blackout'),
('blink', 'blink'),
('blinks', 'blink'),
('blue', 'blue'),
('bowel', 'bowel'),
('bowels', 'bowel'),
('broadcasting', 'broadcasting'),
('calm', 'calm'),
('calmed', 'calm'),
('calmer', 'calm'),
('calming', 'calm'),
('calms', 'calm'),
('cardiovascular', 'cardiovascular'),
('chest', 'chest'),
('chill', 'chill'),
('chilled', 'chill'),
('chills', 'chill'),
('cigarette', 'cigaret'),
('cigarettes', 'cigaret'),
('circadian', 'circadian'),
('cognitive', 'cognitive'),
('compare', 'compare'),
('compared', 'compare'),
('compares', 'compare'),
('comparing', 'compare'),
('compliance', 'compliance'),
('compulsive', 'compulsive'),
('concentration', 'concentration'),
('concentrations', 'concentration'),
('concern', 'concern'),
('concerns', 'concern'),
('constricted', 'constricted'),
('content', 'content'),
('continuity', 'continuity'),
('control', 'control'),
('controled', 'control'),
('controling', 'control'),
('controls', 'control'),
('conversation', 'conversation'),
('conversations', 'conversation'),
('conviction', 'conviction'),
('convictions', 'conviction'),
('cortisol', 'cortisol'),
('count', 'count'),
('counted', 'count'),
('counting', 'count'),
('counts', 'count'),
('couple', 'couple'),
('coupled', 'couple'),
('couples', 'couple'),
('cpap', 'cpap'),
('craving', 'craving'),
('cravings', 'craving'),
('current', 'current'),
('cut', 'cut'),
('cuts', 'cut'),
('decrease', 'decrease'),
('decreased', 'decrease'),
('decreases', 'decrease'),
('decreasing', 'decrease'),
('deficit', 'deficit'),
('deficits', 'deficit'),
('delusion', 'delusion'),
('delusional', 'delusional'),
('delusions', 'delusion'),
('dependance', 'dependance'),
('dependence', 'dependance'),
('dependences', 'dependance'),
('depressed', 'depressed'),
('depressive', 'depressive'),
('deprivation', 'deprivation'),
('diaphoretic', 'diaphoretic'),
('diarrhea', 'diarrhea'),
('difficulties', 'difficulty'),
('difficulty', 'difficulty'),
('disinhibit', 'disinhibit'),
('disinhibited', 'disinhibit'),
('disorder', 'disorder'),
('disorders', 'disorder'),
('disorganised', 'disorganise'),
('disorganization', 'disorganisation'),
('disorganize', 'disorganise'),
('disorganized', 'disorganise'),
('distracted', 'distracted'),
('distractible', 'distractible'),
('dizziness', 'dizziness'),
('dizzyness', 'dizziness'),
('downing', 'down'),
('downs', 'down'),
('drank', 'drink'),
('dream', 'dream'),
('dreamed', 'dream'),
('dreaming', 'dream'),
('dreams', 'dream'),
('drink', 'drink'),
('drinking', 'drink'),
('drinks', 'drink'),
('drug', 'drug'),
('drugging', 'drug'),
('drugs', 'drug'),
('duration', 'duration'),
('dysphoric', 'dysphoric'),
('dysthymic', 'dysthymic'),
('eeg', 'eeg'),
('eegs', 'eeg'),
('effect', 'effect'),
('effected', 'effect'),
('effecting', 'effect'),
('effects', 'effect'),
('elicit', 'elicit'),
('elicited', 'elicit'),
('eliciting', 'elicit'),
('elicits', 'elicit'),
('emotional', 'emotional'),
('empathic', 'empathic'),
('employ', 'employ'),
('employed', 'employ'),
('employing', 'employ'),
('employs', 'employ'),
('energies', 'energy'),
('energy', 'energy'),
('engage', 'engage'),
('engaged', 'engaged'),
('engages', 'engage'),
('engaging', 'engage'),
('episode', 'episode'),
('episodes', 'episode'),
('etoh', 'etoh'),
('executive', 'executive'),
('exercise', 'exercise'),
('exercised', 'exercise'),
('exercises', 'exercise'),
('exercising', 'exercise'),
('exercizes', 'exercise'),
('express', 'express'),
('expressed', 'express'),
('expresses', 'express'),
('expressing', 'express'),
('expressive', 'expressive'),
('factor', 'factor'),
('factored', 'factor'),
('factors', 'factor'),
('fast', 'fast'),
('fasted', 'fast'),
('faster', 'fast'),
('fastest', 'fast'),
('fasting', 'fast'),
('fatigue', 'fatigue'),
('fatigued', 'fatigue'),
('fearful', 'fearful'),
('feature', 'feature'),
('featured', 'feature'),
('features', 'feature'),
('featuring', 'feature'),
('feelings', 'feelings'),
('ferritin', 'ferritin'),
('fever', 'fever'),
('fevers', 'fever'),
('financial', 'financial'),
('flat', 'flat'),
('flight', 'flight'),
('flights', 'flight'),
('forget', 'forget'),
('forgetful', 'forgetful'),
('forgets', 'forget'),
('forgetting', 'forget'),
('forgot', 'forget'),
('forgotten', 'forget'),
('fragmentation', 'fragmentation'),
('frighten', 'frighten'),
('frightened', 'frighten'),
('frightens', 'frighten'),
('function', 'function'),
('functioned', 'function'),
('functioning', 'function'),
('functions', 'function'),
('gain', 'gain'),
('gained', 'gain'),
('gaining', 'gain'),
('gamble', 'gamble'),
('gambled', 'gamble'),
('gambles', 'gamble'),
('gambling', 'gamble'),
('general', 'general'),
('grandiose', 'grandiose');

INSERT INTO cqh_normalize (raw, canon) VALUES
('grandiosity', 'grandiosity'),
('guilt', 'guilt'),
('guilty', 'guilty'),
('hallucinate', 'hallucinate'),
('hallucinated', 'hallucinate'),
('hallucinating', 'hallucinate'),
('hallucination', 'hallucination'),
('hallucinations', 'hallucination'),
('head', 'head'),
('headache', 'headache'),
('headaches', 'headache'),
('heads', 'head'),
('heavier', 'heavy'),
('heaviest', 'heavy'),
('heavy', 'heavy'),
('helpless', 'helpless'),
('heroin', 'heroin'),
('high', 'high'),
('higher', 'high'),
('highest', 'high'),
('highs', 'high'),
('hoard', 'hoard'),
('hoarded', 'hoard'),
('hoarding', 'hoard'),
('hoards', 'hoard'),
('hopeful', 'hopeful'),
('hopelessness', 'hopelessness'),
('hormone', 'hormone'),
('hormones', 'hormone'),
('hour', 'hour'),
('hours', 'hour'),
('hypersomnia', 'hypersomnia'),
('hypervigilant', 'hypervigilant'),
('idea', 'idea'),
('ideas', 'idea'),
('ideation', 'ideation'),
('ideations', 'ideation'),
('identified', 'identify'),
('identifies', 'identify'),
('identify', 'identify'),
('identifying', 'identify'),
('illusion', 'illusion'),
('illusions', 'illusion'),
('impairment', 'impairment'),
('impairments', 'impairment'),
('impulse', 'impulse'),
('impulses', 'impulse'),
('impulsive', 'impulsive'),
('inappropriate', 'inappropriate'),
('include', 'include'),
('includes', 'include'),
('incoherent', 'incoherent'),
('indication', 'indication'),
('indications', 'indication'),
('induce', 'induce'),
('induced', 'induce'),
('induces', 'induce'),
('inducing', 'induce'),
('information', 'information'),
('ingestion', 'ingestion'),
('ingestions', 'ingestion'),
('injurious', 'injurious'),
('insertion', 'insertion'),
('insertions', 'insertion'),
('insomnia', 'insomnia'),
('intact', 'intact'),
('intake', 'intake'),
('intakes', 'intake'),
('interest', 'interest'),
('interests', 'interest'),
('interview', 'interview'),
('interviewed', 'interview'),
('interviewing', 'interview'),
('interviews', 'interview'),
('intoxicated', 'intoxicate'),
('intoxication', 'intoxication'),
('irritability', 'irritability'),
('irritable', 'irritable'),
('isolated', 'isolated'),
('language', 'language'),
('languaged', 'language'),
('languages', 'language'),
('latencies', 'latency'),
('latency', 'latency'),
('lesion', 'lesion'),
('lesioning', 'lesion'),
('lesions', 'lesion'),
('liver', 'liver'),
('living', 'living'),
('lonelier', 'lonely'),
('lonely', 'lonely'),
('lose', 'lose'),
('loses', 'lose'),
('losing', 'lose'),
('lost', 'lose'),
('major', 'major'),
('majored', 'major'),
('majoring', 'major'),
('majors', 'major'),
('meaningless', 'meaningless'),
('melatonin', 'melatonin'),
('memories', 'memory'),
('memory', 'memory'),
('mildly', 'mildly'),
('minutes', 'minutes'),
('misinterpret', 'misinterpret'),
('misinterpreted', 'misinterpret'),
('misinterpreting', 'misinterpret'),
('misinterprets', 'misinterpret'),
('misunderstand', 'misunderstand'),
('misunderstands', 'misunderstand'),
('misunderstood', 'misunderstand'),
('money', 'money'),
('monitoring', 'monitoring'),
('mood', 'mood'),
('moods', 'mood'),
('motivate', 'motivate'),
('motivated', 'motivate'),
('motivating', 'motivate'),
('motor', 'motor'),
('movement', 'movement'),
('movements', 'movement'),
('named', 'name'),
('names', 'name'),
('naming', 'name'),
('nausea', 'nausea'),
('nauseas', 'nausea'),
('neck', 'neck'),
('necks', 'neck'),
('needed', 'need'),
('needing', 'need'),
('nervous', 'nervous'),
('night', 'night'),
('nightmare', 'nightmare'),
('nightmares', 'nightmare'),
('nights', 'night'),
('notice', 'notice'),
('noticed', 'notice'),
('notices', 'notice'),
('noticing', 'notice'),
('offer', 'offer'),
('offered', 'offer'),
('offers', 'offer'),
('ongoing', 'ongoing'),
('optimistic', 'optimistic'),
('oral', 'oral'),
('ordering', 'ordering'),
('osa', 'osa'),
('outburst', 'outburst'),
('outbursts', 'outburst'),
('pain', 'pain'),
('pained', 'pain'),
('pains', 'pain'),
('palpitation', 'palpitation'),
('palpitations', 'palpitation'),
('panic', 'panic'),
('panicked', 'panic'),
('panicking', 'panic'),
('panics', 'panic'),
('paranoid', 'paranoid'),
('paranoids', 'paranoid'),
('partial', 'partial'),
('perception', 'perception'),
('perceptions', 'perception'),
('pessimistic', 'pessimistic'),
('phase', 'faze'),
('phases', 'faze'),
('phasing', 'faze'),
('phobia', 'phobia'),
('phobias', 'phobia'),
('phobic', 'phobic'),
('planning', 'planning'),
('possibly', 'possibly'),
('posttraumatic', 'posttraumatic'),
('problem', 'problem'),
('problems', 'problem'),
('prominent', 'prominent'),
('psychomotor', 'psychomotor'),
('psychotic', 'psychotic'),
('psychotics', 'psychotic'),
('quantities', 'quantity'),
('quantity', 'quantity'),
('question', 'question'),
('questioned', 'question'),
('questions', 'question'),
('race', 'race'),
('raced', 'race'),
('races', 'race'),
('racing', 'race'),
('range', 'range'),
('ranged', 'range'),
('ranges', 'range'),
('ranging', 'range'),
('reactivity', 'reactivity'),
('recall', 'recall'),
('recalled', 'recall'),
('recalling', 'recall'),
('recalls', 'recall'),
('refuse', 'refuse'),
('refused', 'refuse'),
('refuses', 'refuse'),
('refusing', 'refuse'),
('regular', 'regular'),
('regularly', 'regularly'),
('related', 'related'),
('relaxed', 'relaxed'),
('rem', 'rem'),
('remission', 'remission'),
('remissions', 'remission'),
('repetitive', 'repetitive'),
('report', 'report'),
('reported', 'report'),
('reports', 'report'),
('resource', 'resource'),
('resources', 'resource'),
('respiratory', 'respiratory'),
('respond', 'respond'),
('responded', 'respond'),
('responding', 'respond'),
('responds', 'respond'),
('restless', 'restless'),
('restoration', 'restoration'),
('rhythm', 'rhythm'),
('rhythms', 'rhythm'),
('routine', 'routine'),
('routines', 'routine'),
('sad', 'sad'),
('scare', 'scare'),
('scared', 'scare'),
('scares', 'scare'),
('scaring', 'scare'),
('schizophrenia', 'schizophrenia'),
('school', 'school'),
('schooled', 'school'),
('schooling', 'school'),
('schools', 'school'),
('sensation', 'sensation'),
('sensations', 'sensation'),
('sensitivities', 'sensitivity'),
('sensitivity', 'sensitivity'),
('sensory', 'sensory'),
('sexual', 'sexual'),
('situation', 'situation'),
('situations', 'situation'),
('sleep', 'sleep'),
('sleepiness', 'sleepiness'),
('sleeping', 'sleep'),
('sleeps', 'sleep'),
('slept', 'sleep'),
('spindle', 'spindle');

INSERT INTO cqh_normalize (raw, canon) VALUES
('spindles', 'spindle'),
('stability', 'stability'),
('stable', 'stable'),
('stables', 'stable'),
('startle', 'startle'),
('startled', 'startle'),
('startles', 'startle'),
('stereotypies', 'stereotypy'),
('stereotypy', 'stereotypy'),
('stop', 'stop'),
('stopped', 'stop'),
('stopping', 'stop'),
('stops', 'stop'),
('strategies', 'strategy'),
('strategy', 'strategy'),
('stress', 'stress'),
('stressed', 'stress'),
('stresses', 'stress'),
('stressing', 'stress'),
('stressor', 'stressor'),
('stressors', 'stressor'),
('studies', 'study'),
('study', 'study'),
('studying', 'study'),
('suicidal', 'suicidal'),
('sweat', 'sweat'),
('sweated', 'sweat'),
('sweating', 'sweat'),
('sweats', 'sweat'),
('symptom', 'symptom'),
('symptoms', 'symptom'),
('tachycardic', 'tachycardic'),
('talk', 'talk'),
('talked', 'talk'),
('talking', 'talk'),
('talks', 'talk'),
('tearful', 'tearful'),
('tense', 'tense'),
('tensing', 'tense'),
('term', 'term'),
('termed', 'term'),
('terms', 'term'),
('testing', 'testing'),
('testings', 'testing'),
('theme', 'theme'),
('themes', 'theme'),
('therapies', 'therapy'),
('therapy', 'therapy'),
('theta', 'theta'),
('thinking', 'think'),
('thinks', 'think'),
('thought', 'thought'),
('thoughts', 'thought'),
('threaten', 'threaten'),
('threatened', 'threaten'),
('threatening', 'threaten'),
('threatens', 'threaten'),
('timing', 'timing'),
('timings', 'timing'),
('tingle', 'tingle'),
('tingling', 'tingle'),
('tired', 'tired'),
('trauma', 'trauma'),
('traumas', 'trauma'),
('traumata', 'trauma'),
('traumatic', 'traumatic'),
('tremulous', 'tremulous'),
('type', 'type'),
('types', 'type'),
('ultimately', 'ultimately'),
('understand', 'understand'),
('understanding', 'understanding'),
('understandings', 'understanding'),
('understands', 'understand'),
('understood', 'understand'),
('user', 'user'),
('users', 'user'),
('vague', 'vague'),
('variability', 'variability'),
('vigilance', 'vigilance'),
('violence', 'violence'),
('violent', 'violent'),
('visual', 'visual'),
('voice', 'voice'),
('voices', 'voice'),
('voicing', 'voice'),
('voluntary', 'voluntary'),
('vomiting', 'vomiting'),
('wakefulness', 'wakefulness'),
('waking', 'waking'),
('wakings', 'waking'),
('wall', 'wall'),
('walls', 'wall'),
('weight', 'weight'),
('weighted', 'weight'),
('weights', 'weight'),
('withdrawal', 'withdrawal'),
('withdrawals', 'withdrawal'),
('withdrawn', 'withdrawn'),
('word', 'word'),
('worded', 'word'),
('wording', 'word'),
('words', 'word'),
('worried', 'worried'),
('worsen', 'worsen'),
('worsened', 'worsen'),
('worsening', 'worsen'),
('worsens', 'worsen'),
('worth', 'worth'),
('worthless', 'worthless');

INSERT INTO cqh_domain_terms (domain, term) VALUES
('negative', 'aggression'),
('negative', 'aggressive'),
('negative', 'agitated'),
('negative', 'anger'),
('negative', 'angry'),
('negative', 'anxiety'),
('negative', 'anxiety_disorder'),
('negative', 'anxious'),
('negative', 'appetite'),
('negative', 'ashamed'),
('negative', 'assaultive'),
('negative', 'attack'),
('negative', 'avoidance'),
('negative', 'avoidant'),
('negative', 'blue'),
('negative', 'calm'),
('negative', 'chest'),
('negative', 'chest_pain'),
('negative', 'chill'),
('negative', 'constricted'),
('negative', 'depressed'),
('negative', 'depressed_mood'),
('negative', 'depressive'),
('negative', 'depressive_disorder'),
('negative', 'diaphoretic'),
('negative', 'diarrhea'),
('negative', 'difficulty'),
('negative', 'down'),
('negative', 'dysphoric'),
('negative', 'dysthymic'),
('negative', 'emotional'),
('negative', 'energy'),
('negative', 'fatigue'),
('negative', 'fearful'),
('negative', 'feelings'),
('negative', 'fever'),
('negative', 'frighten'),
('negative', 'gain'),
('negative', 'guilt'),
('negative', 'guilty'),
('negative', 'head'),
('negative', 'helpless'),
('negative', 'hopeful'),
('negative', 'hopelessness'),
('negative', 'hypervigilant'),
('negative', 'irritable'),
('negative', 'lesion'),
('negative', 'living'),
('negative', 'major'),
('negative', 'meaningless'),
('negative', 'nausea'),
('negative', 'nervous'),
('negative', 'optimistic'),
('negative', 'outburst'),
('negative', 'palpitation'),
('negative', 'panic'),
('negative', 'panic_attack'),
('negative', 'panic_disorder'),
('negative', 'pessimistic'),
('negative', 'phobia'),
('negative', 'phobic'),
('negative', 'posttraumatic'),
('negative', 'posttraumatic_stress'),
('negative', 'refuse'),
('negative', 'relaxed'),
('negative', 'respond'),
('negative', 'sad'),
('negative', 'scare'),
('negative', 'sensory'),
('negative', 'sensory_deficit'),
('negative', 'situation'),
('negative', 'startle'),
('negative', 'stress'),
('negative', 'stress_disorder'),
('negative', 'stressor'),
('negative', 'suicidal'),
('negative', 'sweat'),
('negative', 'tachycardic'),
('negative', 'tearful'),
('negative', 'tense'),
('negative', 'threaten'),
('negative', 'trauma'),
('negative', 'tremulous'),
('negative', 'violence'),
('negative', 'violent'),
('negative', 'weight_gain'),
('negative', 'worried'),
('negative', 'worth'),
('negative', 'worthless'),
('positive', 'abuse'),
('positive', 'alcohol'),
('positive', 'alcohol_abuse'),
('positive', 'alcohol_dependance'),
('positive', 'alcohol_withdrawal'),
('positive', 'appetite'),
('positive', 'back'),
('positive', 'barbiturate'),
('positive', 'blackout'),
('positive', 'bowel'),
('positive', 'compulsive'),
('positive', 'count'),
('positive', 'couple'),
('positive', 'craving'),
('positive', 'cut'),
('positive', 'dependance'),
('positive', 'disorder_nos'),
('positive', 'drink'),
('positive', 'drug'),
('positive', 'effect'),
('positive', 'etoh'),
('positive', 'financial'),
('positive', 'gamble'),
('positive', 'heavy'),
('positive', 'heroin'),
('positive', 'high'),
('positive', 'high_school'),
('positive', 'hoard'),
('positive', 'indication'),
('positive', 'induce'),
('positive', 'induce_mood'),
('positive', 'ingestion'),
('positive', 'intake'),
('positive', 'interest'),
('positive', 'intoxicate'),
('positive', 'intoxication'),
('positive', 'liver'),
('positive', 'lose'),
('positive', 'money'),
('positive', 'monitoring'),
('positive', 'motivate'),
('positive', 'motor'),
('positive', 'movement'),
('positive', 'ordering'),
('positive', 'partial'),
('positive', 'planning'),
('positive', 'psychotic_disorder'),
('positive', 'related'),
('positive', 'remission'),
('positive', 'repetitive'),
('positive', 'report_that'),
('positive', 'resource'),
('positive', 'routine'),
('positive', 'sexual_abuse'),
('positive', 'stereotypy'),
('positive', 'stop'),
('positive', 'strategy'),
('positive', 'therapy'),
('positive', 'with_alcohol'),
('positive', 'withdrawal'),
('positive', 'withdrawal_symptom'),
('cognitive', 'ability'),
('cognitive', 'answer'),
('cognitive', 'aphasic'),
('cognitive', 'attention'),
('cognitive', 'attentive'),
('cognitive', 'auditory'),
('cognitive', 'auditory_hallucination'),
('cognitive', 'behavior'),
('cognitive', 'cognitive'),
('cognitive', 'compliance'),
('cognitive', 'content'),
('cognitive', 'conversation'),
('cognitive', 'conviction'),
('cognitive', 'current_episode'),
('cognitive', 'decrease_need'),
('cognitive', 'delusion'),
('cognitive', 'disinhibit'),
('cognitive', 'disorganisation'),
('cognitive', 'disorganise'),
('cognitive', 'disorganise_behavior'),
('cognitive', 'distracted'),
('cognitive', 'distractible'),
('cognitive', 'employ'),
('cognitive', 'engage'),
('cognitive', 'executive'),
('cognitive', 'exercise'),
('cognitive', 'factor_include'),
('cognitive', 'fast'),
('cognitive', 'flight'),
('cognitive', 'forget'),
('cognitive', 'forgetful'),
('cognitive', 'grandiosity'),
('cognitive', 'hallucinate'),
('cognitive', 'hallucination'),
('cognitive', 'idea'),
('cognitive', 'illusion'),
('cognitive', 'impairment'),
('cognitive', 'impulse'),
('cognitive', 'impulsive'),
('cognitive', 'impulsive_behavior'),
('cognitive', 'inappropriate'),
('cognitive', 'incoherent'),
('cognitive', 'information'),
('cognitive', 'injurious_behavior'),
('cognitive', 'insertion'),
('cognitive', 'intact'),
('cognitive', 'irritable'),
('cognitive', 'memory'),
('cognitive', 'mildly'),
('cognitive', 'name'),
('cognitive', 'ongoing'),
('cognitive', 'perception'),
('cognitive', 'planning'),
('cognitive', 'possibly'),
('cognitive', 'problem'),
('cognitive', 'psychotic'),
('cognitive', 'psychotic_disorder'),
('cognitive', 'psychotic_feature'),
('cognitive', 'question'),
('cognitive', 'race'),
('cognitive', 'recall'),
('cognitive', 'regularly'),
('cognitive', 'related'),
('cognitive', 'schizophrenia'),
('cognitive', 'sensation'),
('cognitive', 'sexual_abuse'),
('cognitive', 'sweat'),
('cognitive', 'talk'),
('cognitive', 'term'),
('cognitive', 'testing'),
('cognitive', 'thought'),
('cognitive', 'thought_insertion'),
('cognitive', 'type'),
('cognitive', 'ultimately'),
('cognitive', 'understand'),
('cognitive', 'understanding'),
('cognitive', 'visual'),
('cognitive', 'visual_hallucination'),
('cognitive', 'voice'),
('cognitive', 'vomiting'),
('cognitive', 'with_psychotic'),
('cognitive', 'word'),
('social', 'aggressive'),
('social', 'auditory'),
('social', 'auditory_hallucination'),
('social', 'behavior'),
('social', 'belief'),
('social', 'compliance'),
('social', 'concern_that'),
('social', 'content'),
('social', 'control'),
('social', 'current_episode'),
('social', 'delusion'),
('social', 'delusional'),
('social', 'disorder_nos'),
('social', 'disorganise'),
('social', 'elicit'),
('social', 'empathic'),
('social', 'engaged'),
('social', 'expressive');

INSERT INTO cqh_domain_terms (domain, term) VALUES
('social', 'fast'),
('social', 'flat'),
('social', 'grandiose'),
('social', 'hallucinate'),
('social', 'hallucination'),
('social', 'impulse'),
('social', 'impulse_control'),
('social', 'inappropriate'),
('social', 'interview'),
('social', 'isolated'),
('social', 'language'),
('social', 'lonely'),
('social', 'misinterpret'),
('social', 'misunderstand'),
('social', 'paranoid'),
('social', 'paranoid_ideation'),
('social', 'partial'),
('social', 'prominent'),
('social', 'psychotic'),
('social', 'psychotic_disorder'),
('social', 'psychotic_feature'),
('social', 'regular'),
('social', 'related'),
('social', 'theme'),
('social', 'think'),
('social', 'thought_broadcasting'),
('social', 'threaten'),
('social', 'type'),
('social', 'vague'),
('social', 'visual'),
('social', 'voice'),
('social', 'wall'),
('social', 'with_psychotic'),
('social', 'withdrawn'),
('social', 'worthless'),
('arousal_regulatory', 'abnormality'),
('arousal_regulatory', 'agitated'),
('arousal_regulatory', 'agitation'),
('arousal_regulatory', 'alertness'),
('arousal_regulatory', 'alfa'),
('arousal_regulatory', 'amount'),
('arousal_regulatory', 'apnea'),
('arousal_regulatory', 'appetite'),
('arousal_regulatory', 'architecture'),
('arousal_regulatory', 'attention'),
('arousal_regulatory', 'bipap'),
('arousal_regulatory', 'blink'),
('arousal_regulatory', 'cardiovascular'),
('arousal_regulatory', 'chest_pain'),
('arousal_regulatory', 'chill'),
('arousal_regulatory', 'cigaret'),
('arousal_regulatory', 'circadian'),
('arousal_regulatory', 'compare'),
('arousal_regulatory', 'concentration'),
('arousal_regulatory', 'continuity'),
('arousal_regulatory', 'cortisol'),
('arousal_regulatory', 'cpap'),
('arousal_regulatory', 'deficit'),
('arousal_regulatory', 'deprivation'),
('arousal_regulatory', 'difficulty'),
('arousal_regulatory', 'dizziness'),
('arousal_regulatory', 'dream'),
('arousal_regulatory', 'duration'),
('arousal_regulatory', 'eeg'),
('arousal_regulatory', 'emotional'),
('arousal_regulatory', 'energy'),
('arousal_regulatory', 'executive_function'),
('arousal_regulatory', 'express'),
('arousal_regulatory', 'fatigue'),
('arousal_regulatory', 'faze'),
('arousal_regulatory', 'feelings'),
('arousal_regulatory', 'ferritin'),
('arousal_regulatory', 'fragmentation'),
('arousal_regulatory', 'have_been'),
('arousal_regulatory', 'headache'),
('arousal_regulatory', 'hormone'),
('arousal_regulatory', 'hour'),
('arousal_regulatory', 'hypersomnia'),
('arousal_regulatory', 'identify'),
('arousal_regulatory', 'information'),
('arousal_regulatory', 'insomnia'),
('arousal_regulatory', 'irritability'),
('arousal_regulatory', 'language'),
('arousal_regulatory', 'melatonin'),
('arousal_regulatory', 'memory'),
('arousal_regulatory', 'minutes'),
('arousal_regulatory', 'motivate'),
('arousal_regulatory', 'motor'),
('arousal_regulatory', 'motor_activity'),
('arousal_regulatory', 'neck'),
('arousal_regulatory', 'night'),
('arousal_regulatory', 'nightmare'),
('arousal_regulatory', 'notice'),
('arousal_regulatory', 'offer'),
('arousal_regulatory', 'oral'),
('arousal_regulatory', 'osa'),
('arousal_regulatory', 'posttraumatic'),
('arousal_regulatory', 'posttraumatic_stress'),
('arousal_regulatory', 'psychomotor'),
('arousal_regulatory', 'psychomotor_agitation'),
('arousal_regulatory', 'quantity'),
('arousal_regulatory', 'range'),
('arousal_regulatory', 'reactivity'),
('arousal_regulatory', 'rem_sleep'),
('arousal_regulatory', 'respiratory'),
('arousal_regulatory', 'restless'),
('arousal_regulatory', 'restoration'),
('arousal_regulatory', 'rhythm'),
('arousal_regulatory', 'sensation'),
('arousal_regulatory', 'sensitivity'),
('arousal_regulatory', 'sensory'),
('arousal_regulatory', 'sensory_deficit'),
('arousal_regulatory', 'sexual_abuse'),
('arousal_regulatory', 'sleep_latency'),
('arousal_regulatory', 'sleep_study'),
('arousal_regulatory', 'sleepiness'),
('arousal_regulatory', 'someone_else'),
('arousal_regulatory', 'spindle'),
('arousal_regulatory', 'stability'),
('arousal_regulatory', 'stable'),
('arousal_regulatory', 'startle'),
('arousal_regulatory', 'stress'),
('arousal_regulatory', 'stress_disorder'),
('arousal_regulatory', 'symptom_include'),
('arousal_regulatory', 'these_problem'),
('arousal_regulatory', 'theta'),
('arousal_regulatory', 'timing'),
('arousal_regulatory', 'tingle'),
('arousal_regulatory', 'tired'),
('arousal_regulatory', 'trauma'),
('arousal_regulatory', 'traumatic'),
('arousal_regulatory', 'user'),
('arousal_regulatory', 'variability'),
('arousal_regulatory', 'vigilance'),
('arousal_regulatory', 'voluntary'),
('arousal_regulatory', 'wakefulness'),
('arousal_regulatory', 'waking'),
('arousal_regulatory', 'weight_gain'),
('arousal_regulatory', 'with_general'),
('arousal_regulatory', 'worsen');

INSERT INTO cqh_all_terms (term) VALUES
('ability'),
('abnormality'),
('abuse'),
('aggression'),
('aggressive'),
('agitated'),
('agitation'),
('alcohol'),
('alcohol_abuse'),
('alcohol_dependance'),
('alcohol_withdrawal'),
('alertness'),
('alfa'),
('amount'),
('anger'),
('angry'),
('answer'),
('anxiety'),
('anxiety_disorder'),
('anxious'),
('aphasic'),
('apnea'),
('appetite'),
('architecture'),
('ashamed'),
('assaultive'),
('attack'),
('attention'),
('attentive'),
('auditory'),
('auditory_hallucination'),
('avoidance'),
('avoidant'),
('back'),
('barbiturate'),
('behavior'),
('belief'),
('bipap'),
('blackout'),
('blink'),
('blue'),
('bowel'),
('calm'),
('cardiovascular'),
('chest'),
('chest_pain'),
('chill'),
('cigaret'),
('circadian'),
('cognitive'),
('compare'),
('compliance'),
('compulsive'),
('concentration'),
('concern_that'),
('constricted'),
('content'),
('continuity'),
('control'),
('conversation'),
('conviction'),
('cortisol'),
('count'),
('couple'),
('cpap'),
('craving'),
('current_episode'),
('cut'),
('decrease_need'),
('deficit'),
('delusion'),
('delusional'),
('dependance'),
('depressed'),
('depressed_mood'),
('depressive'),
('depressive_disorder'),
('deprivation'),
('diaphoretic'),
('diarrhea'),
('difficulty'),
('disinhibit'),
('disorder_nos'),
('disorganisation'),
('disorganise'),
('disorganise_behavior'),
('distracted'),
('distractible'),
('dizziness'),
('down'),
('dream'),
('drink'),
('drug'),
('duration'),
('dysphoric'),
('dysthymic'),
('eeg'),
('effect'),
('elicit'),
('emotional'),
('empathic'),
('employ'),
('energy'),
('engage'),
('engaged'),
('etoh'),
('executive'),
('executive_function'),
('exercise'),
('express'),
('expressive'),
('factor_include'),
('fast'),
('fatigue'),
('faze'),
('fearful'),
('feelings'),
('ferritin'),
('fever'),
('financial'),
('flat'),
('flight'),
('forget'),
('forgetful'),
('fragmentation'),
('frighten'),
('gain'),
('gamble'),
('grandiose'),
('grandiosity'),
('guilt'),
('guilty'),
('hallucinate'),
('hallucination'),
('have_been'),
('head'),
('headache'),
('heavy'),
('helpless'),
('heroin'),
('high'),
('high_school'),
('hoard'),
('hopeful'),
('hopelessness'),
('hormone'),
('hour'),
('hypersomnia'),
('hypervigilant'),
('idea'),
('identify'),
('illusion'),
('impairment'),
('impulse'),
('impulse_control'),
('impulsive'),
('impulsive_behavior'),
('inappropriate'),
('incoherent'),
('indication'),
('induce'),
('induce_mood'),
('information'),
('ingestion'),
('injurious_behavior'),
('insertion'),
('insomnia'),
('intact'),
('intake'),
('interest'),
('interview'),
('intoxicate'),
('intoxication'),
('irritability'),
('irritable'),
('isolated'),
('language'),
('lesion'),
('liver'),
('living'),
('lonely'),
('lose'),
('major'),
('meaningless'),
('melatonin'),
('memory'),
('mildly'),
('minutes'),
('misinterpret'),
('misunderstand'),
('money'),
('monitoring'),
('motivate'),
('motor'),
('motor_activity'),
('movement'),
('name'),
('nausea'),
('neck'),
('nervous'),
('night'),
('nightmare'),
('notice'),
('offer'),
('ongoing'),
('optimistic'),
('oral'),
('ordering'),
('osa'),
('outburst'),
('palpitation'),
('panic'),
('panic_attack'),
('panic_disorder'),
('paranoid'),
('paranoid_ideation'),
('partial'),
('perception'),
('pessimistic'),
('phobia'),
('phobic'),
('planning'),
('possibly'),
('posttraumatic'),
('posttraumatic_stress'),
('problem'),
('prominent'),
('psychomotor'),
('psychomotor_agitation'),
('psychotic'),
('psychotic_disorder'),
('psychotic_feature'),
('quantity'),
('question'),
('race'),
('range'),
('reactivity'),
('recall'),
('refuse'),
('regular'),
('regularly'),
('related'),
('relaxed'),
('rem_sleep'),
('remission'),
('repetitive'),
('report_that'),
('resource'),
('respiratory'),
('respond');

INSERT INTO cqh_all_terms (term) VALUES
('restless'),
('restoration'),
('rhythm'),
('routine'),
('sad'),
('scare'),
('schizophrenia'),
('sensation'),
('sensitivity'),
('sensory'),
('sensory_deficit'),
('sexual_abuse'),
('situation'),
('sleep_latency'),
('sleep_study'),
('sleepiness'),
('someone_else'),
('spindle'),
('stability'),
('stable'),
('startle'),
('stereotypy'),
('stop'),
('strategy'),
('stress'),
('stress_disorder'),
('stressor'),
('suicidal'),
('sweat'),
('symptom_include'),
('tachycardic'),
('talk'),
('tearful'),
('tense'),
('term'),
('testing'),
('theme'),
('therapy'),
('these_problem'),
('theta'),
('think'),
('thought'),
('thought_broadcasting'),
('thought_insertion'),
('threaten'),
('timing'),
('tingle'),
('tired'),
('trauma'),
('traumatic'),
('tremulous'),
('type'),
('ultimately'),
('understand'),
('understanding'),
('user'),
('vague'),
('variability'),
('vigilance'),
('violence'),
('violent'),
('visual'),
('visual_hallucination'),
('voice'),
('voluntary'),
('vomiting'),
('wakefulness'),
('waking'),
('wall'),
('weight_gain'),
('with_alcohol'),
('with_general'),
('with_psychotic'),
('withdrawal'),
('withdrawal_symptom'),
('withdrawn'),
('word'),
('worried'),
('worsen'),
('worth'),
('worthless');

CREATE VIEW cqh_notes_source AS
SELECT
  id AS note_id,
  note_text AS raw_text
FROM notes;

CREATE VIEW cqh_notes_cleaned AS
SELECT
  note_id,
  trim(replace(replace(replace(replace(replace(replace(replace(replace(replace(replace(replace(replace(replace(replace(replace(replace(replace(replace(replace(replace(replace(replace(replace(replace(replace(lower(coalesce(raw_text, '')), '.', ' '), ',', ' '), ';', ' '), ':', ' '), '!', ' '), '?', ' '), '(', ' '), ')', ' '), '[', ' '), ']', ' '), '{', ' '), '}', ' '), '/', ' '), '\', ' '), '-', ' '), '_', ' '), '"', ' '), '''', ' '), '&', ' '), '+', ' '), '=', ' '), '*', ' '), char(10), ' '), char(13), ' '), char(9), ' ')) AS cleaned_text
FROM cqh_notes_source;

-- Tokenize via json_each to avoid the recursive-CTE recursion limit.
-- Safe because build_clean_expr strips double quotes and backslashes from
-- cleaned_text, so nothing inside the array literal can break JSON parsing.
CREATE VIEW cqh_note_tokens_raw AS
SELECT
  nc.note_id,
  je.key + 1 AS pos,
  je.value   AS token_raw
FROM cqh_notes_cleaned nc,
     json_each('["' || replace(nc.cleaned_text, ' ', '","') || '"]') je
WHERE je.value <> '';

CREATE VIEW cqh_note_tokens AS
SELECT
  filtered.note_id,
  row_number() OVER (PARTITION BY filtered.note_id ORDER BY filtered.raw_pos) AS pos,
  coalesce(n.canon, filtered.token_raw) AS token
FROM (
  SELECT
    t.note_id,
    t.pos AS raw_pos,
    t.token_raw
  FROM cqh_note_tokens_raw t
  LEFT JOIN cqh_stopwords s
    ON s.word = t.token_raw
  WHERE length(t.token_raw) > 2
    AND t.token_raw GLOB '*[^0-9]*'
    AND s.word IS NULL
) AS filtered
LEFT JOIN cqh_normalize n
  ON n.raw = filtered.token_raw;

CREATE VIEW cqh_note_term_counts AS
WITH token_pairs AS (
  SELECT
    note_id,
    pos,
    token,
    lead(token) OVER (PARTITION BY note_id ORDER BY pos) AS next_token
  FROM cqh_note_tokens
),
term_hits AS (
  SELECT
    p.note_id,
    p.token AS term
  FROM token_pairs p
  JOIN cqh_all_terms a
    ON a.term = p.token
  UNION ALL
  SELECT
    p.note_id,
    p.token || '_' || p.next_token AS term
  FROM token_pairs p
  JOIN cqh_all_terms a
    ON a.term = p.token || '_' || p.next_token
  WHERE p.next_token IS NOT NULL
)
SELECT
  note_id,
  term,
  count(*) AS hit_count
FROM term_hits
GROUP BY note_id, term;

-- Only emit rows for (note, domain) pairs with at least one match;
-- cqh_note_scores coalesces missing pairs to 0.0.
CREATE VIEW cqh_note_domain_hits AS
WITH domain_totals AS (
  SELECT
    domain,
    count(*) AS total_terms
  FROM cqh_domain_terms
  GROUP BY domain
)
SELECT
  tc.note_id,
  dt.domain,
  count(*) AS matched_terms,
  totals.total_terms
FROM cqh_note_term_counts tc
JOIN cqh_domain_terms dt
  ON dt.term = tc.term
JOIN domain_totals totals
  ON totals.domain = dt.domain
WHERE tc.hit_count > 0
GROUP BY tc.note_id, dt.domain, totals.total_terms;

CREATE VIEW cqh_note_scores AS
WITH note_lengths AS (
  SELECT
    ns.note_id,
    count(t.token) AS length
  FROM cqh_notes_source ns
  LEFT JOIN cqh_note_tokens t
    ON t.note_id = ns.note_id
  GROUP BY ns.note_id
),
note_matches AS (
  SELECT
    ns.note_id,
    coalesce(sum(tc.hit_count), 0) AS match_count
  FROM cqh_notes_source ns
  LEFT JOIN cqh_note_term_counts tc
    ON tc.note_id = ns.note_id
  GROUP BY ns.note_id
),
domain_scores AS (
  SELECT
    note_id,
    domain,
    round(CASE
      WHEN total_terms = 0 THEN 0.0
      ELSE 1.0 * matched_terms / total_terms
    END, 5) AS score
  FROM cqh_note_domain_hits
)
SELECT
  ns.note_id,
  coalesce(nl.length, 0) AS length,
  coalesce(nm.match_count, 0) AS match_count,
  coalesce(max(CASE WHEN ds.domain = 'negative' THEN ds.score END), 0.0) AS negative,
  coalesce(max(CASE WHEN ds.domain = 'positive' THEN ds.score END), 0.0) AS positive,
  coalesce(max(CASE WHEN ds.domain = 'cognitive' THEN ds.score END), 0.0) AS cognitive,
  coalesce(max(CASE WHEN ds.domain = 'social' THEN ds.score END), 0.0) AS social,
  coalesce(max(CASE WHEN ds.domain = 'arousal_regulatory' THEN ds.score END), 0.0) AS arousal_regulatory
FROM cqh_notes_source ns
LEFT JOIN note_lengths nl
  ON nl.note_id = ns.note_id
LEFT JOIN note_matches nm
  ON nm.note_id = ns.note_id
LEFT JOIN domain_scores ds
  ON ds.note_id = ns.note_id
GROUP BY
  ns.note_id,
  nl.length,
  nm.match_count
ORDER BY ns.note_id;

COMMIT;
)---"

From the shell this would be easier as:

```bash
$ sqlite3 rdoc.db < RDOC_SQL_SCHEMA.sql

$ sqlite3 rdoc.db <<'EOF'
.mode csv
.import sobp_nlp.csv notes_staging
INSERT INTO notes(id, note_text)
  SELECT nid, note FROM notes_staging;
EOF

$ sqlite3 -header -csv rdoc.db "SELECT * FROM cqh_note_scores;" > sql_scores.csv
```

In [7]:
system2("sqlite3", args = "rdoc.db", input = RDOC_SQL_SCHEMA)
#system("sqlite3 rdoc.db <<'EOF'
#.mode csv
#.import sobp_nlp.csv notes_staging
#INSERT INTO notes(id, note_text)
#  SELECT nid, note FROM notes_staging;
#EOF
#")
# system("sqlite3 -header -csv rdoc.db 'SELECT * FROM cqh_note_scores;' > sql_scores.csv")

##### MGH CQH RDoC LLM `score_rdoc_llm`

McCoy TH, Perlis RH. Dimensional Measures of Psychopathology in Children and Adolescents Using Large Language Models. *Biol Psychiatry*. 2024;96(12):940-947. doi: 10.1016/j.biopsych.2024.05.008. PMID: [38866172](https://pubmed.ncbi.nlm.nih.gov/38866172/)

In [8]:
reset_rdoc_ollama_env <- function() {
  Sys.setenv(RDOC_LLM_PROVIDER = "ollama",
            RDOC_LLM_MODEL = "llama3.2:3b")
}
reset_rdoc_ollama_env()

In [9]:
# @title
score_rdoc_llm <- local({

# CQH Dimensional Phenotyper - LLM variant
#
# Reference:
#   McCoy TH, Perlis RH. Dimensional Measures of Psychopathology in Children
#   and Adolescents Using Large Language Models. Biol Psychiatry. 2024 Dec
#   15;96(12):940-947. doi: 10.1016/j.biopsych.2024.05.008. PMID: 38866172.
#
#   McCoy TH, Perlis RH. Characterizing research domain criteria symptoms among
#   psychiatric inpatients using large language models. J Mood Anxiety Disord.
#   2024 Jul 20;8:100079. doi: 10.1016/j.xjmad.2024.100079. PMID: 40655912.
#
# Uses ellmer to estimate RDoC symptom burden from free-text notes, via either
# a local Ollama server or a hosted OpenRouter model. The output is a named
# numeric vector that can be rbound across documents, similar to the matrix
# shape returned by CQHDimensionalPhenotyper.R.
#
# Public API:
#   make_rdoc_chat()               -> ellmer chat object
#   score_document_llm(text)       -> named numeric vector (one document;
#                                     raises on LLM failure after retries)
#   score_documents_llm(texts)     -> data.frame (many documents; per-document
#                                     failures yield a row of NA domain scores
#                                     and a summary warning, rather than
#                                     aborting the whole batch)
#
# Resilience:
#   Each LLM call is attempted up to `max_attempts` times (default 3) with a
#   short exponential backoff between tries before it is counted as a failure.
#   Each attempt is also bounded by `timeout_secs` (default 120) via
#   setTimeLimit(), and the response is capped at `max_tokens = 4096` to keep
#   a doom-looping model from filling the context window. Note that
#   setTimeLimit() fires at R-level checkpoints; a wedged C-level socket read
#   may not unblock until the kernel or the mclapply worker is killed, so the
#   timeout catches the realistic stall cases but is not an absolute cap.
#   In the vector path, documents that still fail return a row whose
#   domain columns are NA (distinct from 0 on the 0-10 scale) and whose
#   `length` and `elapsed_time` are still populated. Structured responses that contain
#   invalid field values also preserve those fields as NA rather than
#   converting them to zero. One summary warning names the failed indices so
#   the caller can retry just those.
#
# Runtime configuration:
#   RDOC_LLM_PROVIDER  env var: "ollama" (default) or "openrouter".
#   RDOC_LLM_MODEL     env var: model id; defaults to "llama3.2" for ollama
#                      and "anthropic/claude-haiku-4.5" for openrouter.
#   OPENROUTER_API_KEY env var: required when provider = "openrouter".
#   OLLAMA_BASE_URL    env var: ollama endpoint; defaults to localhost:11434.
#   `base_url` and `num_ctx` are Ollama-only and ignored for OpenRouter.
#   `workers` (score_documents_llm only): when > 1 and the `parallel`
#              package is available on a non-Windows platform, documents are
#              fanned out via parallel::mclapply(mc.preschedule = FALSE);
#              otherwise the call falls back to serial lapply.
#
# Expected runtime setup (Ollama path):
#   install.packages("ellmer")
#   ollama serve
#   ollama pull llama3.2
#
# Expected runtime setup (OpenRouter path):
#   install.packages("ellmer")
#   Sys.setenv(RDOC_LLM_PROVIDER = "openrouter",
#              RDOC_LLM_MODEL    = "anthropic/claude-haiku-4.5",
#              OPENROUTER_API_KEY = "sk-or-...")

if (!requireNamespace("ellmer", quietly = TRUE)) {
  stop(
    "CQHDimensionalPhenotyperLLM.R requires the 'ellmer' package. ",
    "Install with install.packages('ellmer')."
  )
}

RDOC_LLM_DOMAINS <- c(
  "negative",
  "positive",
  "cognitive",
  "social",
  "arousal",
  "sensorimotor"
)

RDOC_LLM_OUTPUT_COLUMNS <- c(
  "length",
  RDOC_LLM_DOMAINS,
  "elapsed_time"
)

rdoc_llm_system_prompt <- r"(You are a skilled psychiatrist scoring an emergency room clinical note in terms of how the patient symptoms over the past 24 hours reflect the 6 NIMH Research Domain Criteria (RDoC): Negative Valence Systems, Positive Valence Systems, Cognitive Systems, Social Processes, Arousal and Regulatory Systems, and Sensorimotor Systems.

Remember that substance use can be reflected as Positive Valence symptom.

Notes are scored on a 0-10 scale to capture the magnitude of documented symptoms relevant in a given domain. Score 0 if no symptoms are present and functioning is normal, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe. Generally moderate requires treatment and severe requires hospitalization.

Return only a valid JSON object with each score, Negative, Positive, Cognitive, Social, Arousal, and Sensorimotor.)"

rdoc_llm_user_prompt <- function(text) {
  paste("Clinical note:", text, sep = "\n\n")
}

rdoc_llm_schema <- function() {
  ellmer::type_object(
    "RDoC symptom burden scores for a single ER clinical note, 0-10 per domain.",
    negative = ellmer::type_number(
      "Negative Valence Systems, 0-10 inclusive."
    ),
    positive = ellmer::type_number(
      "Positive Valence Systems (includes substance use), 0-10 inclusive."
    ),
    cognitive = ellmer::type_number(
      "Cognitive Systems, 0-10 inclusive."
    ),
    social = ellmer::type_number(
      "Systems for Social Processes, 0-10 inclusive."
    ),
    arousal = ellmer::type_number(
      "Arousal and Regulatory Systems, 0-10 inclusive."
    ),
    sensorimotor = ellmer::type_number(
      "Sensorimotor Systems, 0-10 inclusive."
    )
  )
}

normalize_note <- function(text) {
  if (!is.character(text) || length(text) != 1L || is.na(text) || !nzchar(text)) {
    return("")
  }

  text <- enc2utf8(text)
  text <- gsub("\r\n?", "\n", text, perl = TRUE)
  text <- gsub("[[:space:]]+", " ", text, perl = TRUE)
  trimws(text)
}

note_word_count <- function(text) {
  text <- normalize_note(text)
  if (!nzchar(text)) {
    return(0)
  }

  length(strsplit(text, "\\s+", perl = TRUE)[[1]])
}

empty_llm_score <- function(text = "", elapsed = 0) {
  vals <- c(note_word_count(text), rep(0, length(RDOC_LLM_DOMAINS)), elapsed)
  stats::setNames(vals, RDOC_LLM_OUTPUT_COLUMNS)
}

# Returned when the LLM call fails after all retries. NA_real_ in the domain
# columns signals "we tried and could not score" so callers can distinguish
# scoring failures from genuine zero-burden notes.
failed_llm_score <- function(text = "", elapsed = 0) {
  vals <- c(note_word_count(text),
            rep(NA_real_, length(RDOC_LLM_DOMAINS)),
            elapsed)
  stats::setNames(vals, RDOC_LLM_OUTPUT_COLUMNS)
}

validate_scalar_count <- function(x, arg_name) {
  x_int <- suppressWarnings(as.integer(x))
  if (!is.numeric(x) || length(x) != 1L || is.na(x) || !is.finite(x) ||
      is.na(x_int) || x < 1 || x != x_int) {
    stop("`", arg_name, "` must be a single integer >= 1.", call. = FALSE)
  }

  x_int
}

RDOC_LLM_DEFAULT_MODELS <- c(
  ollama = "llama3.2",
  openrouter = "anthropic/claude-haiku-4.5"
)

resolve_rdoc_model <- function(provider, model) {
  if (is.null(model) || !nzchar(model)) {
    model <- Sys.getenv("RDOC_LLM_MODEL", "")
  }
  if (!nzchar(model)) {
    model <- RDOC_LLM_DEFAULT_MODELS[[provider]]
  }
  if (is.null(model) || !nzchar(model)) {
    stop("No RDoC LLM model resolved for provider '", provider, "'.", call. = FALSE)
  }
  model
}

make_rdoc_chat <- function(provider = Sys.getenv("RDOC_LLM_PROVIDER", "ollama"),
                           model = NULL,
                           base_url = Sys.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
                           system_prompt = rdoc_llm_system_prompt,
                           params = ellmer::params(temperature = 0, seed = 20260411L, max_tokens = 4096L),
                           num_ctx = 32768L) {
  provider <- tolower(provider)
  if (!provider %in% names(RDOC_LLM_DEFAULT_MODELS)) {
    stop("Unknown RDoC LLM provider: '", provider,
         "'. Use 'ollama' or 'openrouter'.", call. = FALSE)
  }
  model <- resolve_rdoc_model(provider, model)

  switch(provider,
    ollama = ellmer::chat_ollama(
      system_prompt = system_prompt,
      base_url = base_url,
      model = model,
      params = params,
      api_args = list(options = list(num_ctx = num_ctx))
    ),
    openrouter = ellmer::chat_openrouter(
      system_prompt = system_prompt,
      model = model,
      params = params
    )
  )
}

fresh_chat <- function(chat) {
  worker <- chat$clone()
  worker$set_turns(list())
  worker
}

parse_domain_score <- function(x) {
  if (!is.numeric(x) || length(x) != 1L || is.na(x) || !is.finite(x)) {
    return(NA_real_)
  }

  min(10, max(0, as.numeric(x)))
}

score_note_llm <- function(text, chat, schema = rdoc_llm_schema(),
                           max_attempts = 3L, timeout_secs = 120L) {
  max_attempts <- validate_scalar_count(max_attempts, "max_attempts")
  timeout_secs <- validate_scalar_count(timeout_secs, "timeout_secs")
  last_err <- NULL

  for (attempt in seq_len(max_attempts)) {
    result <- tryCatch(
      {
        worker <- fresh_chat(chat)
        setTimeLimit(elapsed = timeout_secs, transient = TRUE)
        on.exit(setTimeLimit(), add = TRUE)
        out <- worker$chat_structured(rdoc_llm_user_prompt(text), type = schema)
        setTimeLimit()
        out
      },
      error = function(e) {
        setTimeLimit()
        e
      }
    )

    if (!inherits(result, "error")) {
      return(vapply(RDOC_LLM_DOMAINS,
                    function(nm) parse_domain_score(result[[nm]]),
                    numeric(1)))
    }

    last_err <- result
    if (attempt < max_attempts) {
      Sys.sleep(0.5 * (2 ^ (attempt - 1L)))
    }
  }

  stop(conditionMessage(last_err), call. = FALSE)
}

#' Score a single clinical document with an ellmer-backed LLM.
#'
#' Returns a named numeric vector that can be row-bound with other documents.
#' `length` is note word count and the six domain columns are burden estimates
#' on a 0-10 scale, with `NA` indicating an invalid returned field value for
#' that domain. When `chat` is supplied, `provider`, `model`, `base_url`, and
#' `num_ctx` are ignored. `base_url` and `num_ctx` only apply when
#' `provider = "ollama"`. `max_attempts` bounds the total number of tries
#' against the ellmer structured-output call; `timeout_secs` bounds the
#' wall-clock duration of each individual attempt via setTimeLimit() (a
#' timeout surfaces as a normal error and is subject to retry). This function
#' raises if every attempt fails. For partial-success semantics across many
#' documents, use `score_documents_llm()`.
score_document_llm <- function(text,
                               chat = NULL,
                               provider = Sys.getenv("RDOC_LLM_PROVIDER", "ollama"),
                               model = NULL,
                               base_url = Sys.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
                               num_ctx = 32768L,
                               schema = rdoc_llm_schema(),
                               max_attempts = 3L,
                               timeout_secs = 120L) {
  max_attempts <- validate_scalar_count(max_attempts, "max_attempts")
  timeout_secs <- validate_scalar_count(timeout_secs, "timeout_secs")

  if (!is.character(text) || length(text) != 1L) {
    stop("score_document_llm() expects a length-1 character vector; ",
         "use score_documents_llm() for multiple documents.")
  }

  t0 <- Sys.time()
  note <- normalize_note(text)
  if (!nzchar(note)) {
    return(empty_llm_score(
      note,
      elapsed = as.numeric(difftime(Sys.time(), t0, units = "secs"))
    ))
  }

  total_length <- note_word_count(note)

  if (is.null(chat)) {
    chat <- make_rdoc_chat(provider = provider, model = model,
                           base_url = base_url, num_ctx = num_ctx)
  }

  domains <- tryCatch(
    score_note_llm(note, chat = chat, schema = schema,
                   max_attempts = max_attempts,
                   timeout_secs = timeout_secs),
    error = function(e) {
      stop("LLM scoring failed for the current document: ", conditionMessage(e), call. = FALSE)
    }
  )

  elapsed <- as.numeric(difftime(Sys.time(), t0, units = "secs"))

  stats::setNames(
    c(total_length, domains, elapsed),
    RDOC_LLM_OUTPUT_COLUMNS
  )
}

# Internal wrapper: never throws. On failure returns a failed_llm_score()
# row with attr(row, "rdoc_error") set to the error message, so the batch
# caller can summarize which documents failed and why.
try_score_document_llm <- function(text, chat, schema, max_attempts, timeout_secs) {
  t0 <- Sys.time()
  tryCatch(
    score_document_llm(text, chat = chat, schema = schema,
                       max_attempts = max_attempts,
                       timeout_secs = timeout_secs),
    error = function(e) {
      row <- failed_llm_score(
        if (is.character(text) && length(text) == 1L) text else "",
        elapsed = as.numeric(difftime(Sys.time(), t0, units = "secs"))
      )
      attr(row, "rdoc_error") <- conditionMessage(e)
      row
    }
  )
}

# Append-only JSONL checkpoint for score_documents_llm().
#
# File layout: line 1 is a `_meta` JSON object pinning the run identity
# (texts hash + provider + model + domain set + version). Subsequent lines
# are one-per-completed-document JSON objects with `index` plus the
# RDOC_LLM_OUTPUT_COLUMNS fields. Only fully-valid rows
# (rdoc_row_is_complete()) are appended; failures and partial-NA responses
# are absent so that a restart re-attempts them. Concurrent appends from
# mclapply children rely on POSIX O_APPEND atomicity for writes <= PIPE_BUF
# (4096 bytes) on a local filesystem; our lines are ~150 bytes. NFS and
# Windows are not supported (Windows already serializes via use_parallel).

rdoc_checkpoint_meta <- function(texts, provider, model) {
  list(
    version = 1L,
    hash = rlang::hash(texts),
    provider = provider,
    model = model,
    domains = RDOC_LLM_DOMAINS
  )
}

rdoc_checkpoint_init <- function(path, meta) {
  line <- jsonlite::toJSON(list(`_meta` = meta), auto_unbox = TRUE)
  cat(line, "\n", sep = "", file = path, append = FALSE)
}

rdoc_checkpoint_load <- function(path, expected_meta, n_texts) {
  lines <- readLines(path, warn = FALSE)
  if (length(lines) == 0L) {
    stop("Checkpoint file is empty: ", path, call. = FALSE)
  }

  header <- tryCatch(
    jsonlite::fromJSON(lines[[1L]], simplifyVector = FALSE),
    error = function(e) {
      stop("Checkpoint header is not valid JSON in ", path, ": ",
           conditionMessage(e), call. = FALSE)
    }
  )
  meta <- header$`_meta`
  if (is.null(meta)) {
    stop("Checkpoint header missing `_meta` in ", path, call. = FALSE)
  }
  for (key in c("version", "hash", "provider", "model", "domains")) {
    found <- unname(unlist(meta[[key]]))
    expect <- unname(unlist(expected_meta[[key]]))
    if (!identical(found, expect)) {
      stop("Checkpoint `", key, "` mismatch in ", path,
           ". Expected ", deparse(expect), " but found ", deparse(found),
           ". Use a different `checkpoint_path` or delete the file.",
           call. = FALSE)
    }
  }

  done <- list()
  for (i in seq_along(lines)[-1L]) {
    line <- lines[[i]]
    if (!nzchar(trimws(line))) next
    rec <- tryCatch(
      jsonlite::fromJSON(line, simplifyVector = TRUE),
      error = function(e) NULL
    )
    if (is.null(rec) || is.null(rec$index)) {
      warning("Skipping unparseable checkpoint line ", i, " in ", path,
              call. = FALSE)
      next
    }
    idx <- suppressWarnings(as.integer(rec$index))
    if (is.na(idx) || idx < 1L || idx > n_texts) {
      warning("Skipping checkpoint line ", i, " in ", path,
              " with out-of-range index ", rec$index, call. = FALSE)
      next
    }
    row <- vapply(RDOC_LLM_OUTPUT_COLUMNS,
                  function(nm) {
                    v <- rec[[nm]]
                    if (is.null(v)) NA_real_ else as.numeric(v)
                  },
                  numeric(1))
    done[[as.character(idx)]] <- stats::setNames(row, RDOC_LLM_OUTPUT_COLUMNS)
  }
  done
}

rdoc_checkpoint_append <- function(path, index, row) {
  rec <- as.list(row)
  rec$index <- as.integer(index)
  line <- jsonlite::toJSON(rec, auto_unbox = TRUE, na = "null", digits = NA)
  cat(line, "\n", sep = "", file = path, append = TRUE)
}

rdoc_row_is_complete <- function(row) {
  vals <- row[RDOC_LLM_DOMAINS]
  is.numeric(vals) && all(is.finite(vals))
}

#' Score many clinical documents with an ellmer-backed LLM.
#'
#' When `chat` is supplied, `provider`, `model`, `base_url`, and `num_ctx` are
#' ignored. `base_url` and `num_ctx` only apply when `provider = "ollama"`.
#' `workers > 1` fans the batch out via `parallel::mclapply` on POSIX systems
#' with the `parallel` package installed; otherwise the call runs serially.
#' `max_attempts` bounds the total number of tries per document and
#' `timeout_secs` bounds each attempt's wall-clock duration via
#' setTimeLimit(). A document that still fails
#' after retries does not abort the batch: its row has NA domain columns and
#' a single summary warning is emitted listing the failed indices. Structured
#' responses with invalid field values keep those domain values as `NA`, and
#' failed rows retain `elapsed_time`.
#'
#' `checkpoint_path` enables append-only JSONL durability. When set, every
#' fully-valid row is written to the file as it lands, keyed by absolute
#' index; restarting with the same path skips already-completed indices and
#' only re-attempts failures and missing rows. The checkpoint is pinned to
#' (`texts` hash, `provider`, `model`, domain set) — any mismatch on restart
#' is a hard error. When `chat` is supplied alongside `checkpoint_path`, an
#' explicit `model` is required so the checkpoint records what produced the
#' scores.
score_documents_llm <- function(texts,
                                chat = NULL,
                                provider = Sys.getenv("RDOC_LLM_PROVIDER", "ollama"),
                                model = NULL,
                                base_url = Sys.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
                                num_ctx = 32768L,
                                schema = rdoc_llm_schema(),
                                workers = 1L,
                                max_attempts = 3L,
                                timeout_secs = 120L,
                                checkpoint_path = NULL) {
  workers <- validate_scalar_count(workers, "workers")
  max_attempts <- validate_scalar_count(max_attempts, "max_attempts")
  timeout_secs <- validate_scalar_count(timeout_secs, "timeout_secs")

  if (!is.character(texts)) {
    stop("score_documents_llm() expects a character vector.")
  }

  if (length(texts) == 0L) {
    return(as.data.frame(
      matrix(numeric(0), nrow = 0, ncol = length(RDOC_LLM_OUTPUT_COLUMNS),
             dimnames = list(NULL, RDOC_LLM_OUTPUT_COLUMNS))
    ))
  }

  chat_supplied <- !is.null(chat)

  done <- list()
  if (!is.null(checkpoint_path)) {
    if (chat_supplied && (is.null(model) || !nzchar(model))) {
      stop("`checkpoint_path` requires an explicit `model` argument when ",
           "`chat` is supplied; the model identity must be recorded in the ",
           "checkpoint meta.", call. = FALSE)
    }
    resolved_provider <- tolower(provider)
    if (!resolved_provider %in% names(RDOC_LLM_DEFAULT_MODELS)) {
      stop("Unknown RDoC LLM provider: '", provider,
           "'. Use 'ollama' or 'openrouter'.", call. = FALSE)
    }
    resolved_model <- resolve_rdoc_model(resolved_provider, model)
    meta <- rdoc_checkpoint_meta(texts, resolved_provider, resolved_model)
    if (file.exists(checkpoint_path)) {
      done <- rdoc_checkpoint_load(checkpoint_path, meta, length(texts))
    } else {
      rdoc_checkpoint_init(checkpoint_path, meta)
    }
  }

  if (!chat_supplied) {
    chat <- make_rdoc_chat(provider = provider, model = model,
                           base_url = base_url, num_ctx = num_ctx)
  }

  use_parallel <- workers > 1L &&
    .Platform$OS.type != "windows" &&
    requireNamespace("parallel", quietly = TRUE)

  todo_idx <- setdiff(seq_along(texts), as.integer(names(done)))

  if (!is.null(checkpoint_path)) {
    preview_n <- min(length(todo_idx), 5L)
    preview <- if (preview_n > 0L) {
      p <- paste(todo_idx[seq_len(preview_n)], collapse = ", ")
      if (length(todo_idx) > preview_n) paste0(p, ", ...") else p
    } else {
      "none"
    }
    message(sprintf(
      "checkpoint: %d/%d already complete, scoring %d this run (indices: %s)",
      length(done), length(texts), length(todo_idx), preview
    ))
  }

  score_one <- function(abs_i) {
    row <- try_score_document_llm(texts[[abs_i]], chat = chat,
                                  schema = schema,
                                  max_attempts = max_attempts,
                                  timeout_secs = timeout_secs)
    if (!is.null(checkpoint_path) && rdoc_row_is_complete(row)) {
      rdoc_checkpoint_append(checkpoint_path, abs_i, row)
    }
    row
  }

  batch_t0 <- Sys.time()
  new_rows <- if (use_parallel) {
    parallel::mclapply(todo_idx, score_one,
                       mc.cores = workers,
                       mc.preschedule = FALSE)
  } else {
    lapply(todo_idx, score_one)
  }

  # mclapply can return try-error objects if a worker died before
  # try_score_document_llm's tryCatch ran (e.g. segfault or interrupt).
  # Convert those into failure rows so rbind stays homogeneous. In that path,
  # only parent-observed wall time is available, not exact child runtime.
  batch_elapsed <- as.numeric(difftime(Sys.time(), batch_t0, units = "secs"))
  new_rows <- lapply(seq_along(new_rows), function(j) {
    r <- new_rows[[j]]
    if (inherits(r, "try-error")) {
      row <- failed_llm_score(texts[[todo_idx[[j]]]], elapsed = batch_elapsed)
      attr(row, "rdoc_error") <- paste("worker error:",
                                       trimws(as.character(r)))
      return(row)
    }
    r
  })

  errors <- vapply(new_rows,
                   function(r) {
                     msg <- attr(r, "rdoc_error")
                     if (is.null(msg)) NA_character_ else as.character(msg)
                   },
                   character(1))
  failed_local <- which(!is.na(errors))
  if (length(failed_local) > 0L) {
    failed_abs <- todo_idx[failed_local]
    preview_n <- min(length(failed_abs), 5L)
    preview <- paste(failed_abs[seq_len(preview_n)], collapse = ", ")
    if (length(failed_abs) > preview_n) {
      preview <- paste0(preview, ", ...")
    }
    warning(sprintf(
      "score_documents_llm(): %d of %d documents failed to score (indices: %s). Failed rows have NA domain columns.",
      length(failed_abs), length(texts), preview
    ), call. = FALSE)
  }

  new_rows <- lapply(new_rows, function(r) { attr(r, "rdoc_error") <- NULL; r })

  rows <- vector("list", length(texts))
  for (j in seq_along(todo_idx)) {
    rows[[todo_idx[[j]]]] <- new_rows[[j]]
  }
  for (k in names(done)) {
    rows[[as.integer(k)]] <- done[[k]]
  }

  if (!is.null(checkpoint_path)) {
    message(sprintf(
      "checkpoint: %d loaded, %d newly scored, %d failed",
      length(done),
      length(new_rows) - length(failed_local),
      length(failed_local)
    ))
  }

  as.data.frame(do.call(rbind, rows))
}

score_documents_llm

})

##### MGH CQH PHQ9 LLM `score_phq9_llm`

McCoy TH, Castro VM, Perlis RH. Estimating depression severity in narrative clinical notes using large language models. *J Affect Disord*. 2025;381:270-274. doi: 10.1016/j.jad.2025.04.014. PMID: [40187432](https://pubmed.ncbi.nlm.nih.gov/40187432/).

In [10]:
reset_phq9_ollama_env <- function() {
  Sys.setenv(PHQ9_LLM_PROVIDER = "ollama",
             PHQ9_LLM_MODEL = "llama3.2:3b")
}
reset_phq9_ollama_env()

In [11]:
# @title
score_phq9_llm <- local({

# CQH PHQ/GAD Symptom Scorer - LLM variant
#
# Reference:
#   McCoy TH, Castro VM, Perlis RH. Estimating depression severity in
#   narrative clinical notes using large language models. J Affect Disord.
#   2025 Jul 15;381:270-274. doi: 10.1016/j.jad.2025.04.014. PMID: 40187432.
#
# Uses ellmer to estimate PHQ-9, GAD-7, and related symptom burden from
# outpatient primary care notes, via either a local Ollama server or a hosted
# OpenRouter model. The output is a named numeric vector that can be rbound
# across documents.
#
# Public API:
#   make_phq9_chat()                    -> ellmer chat object
#   score_document_phq9_llm(text)       -> named numeric vector (one document;
#                                          raises on LLM failure after retries)
#   score_documents_phq9_llm(texts)     -> data.frame (many documents;
#                                          per-document failures yield a row
#                                          of NA score fields plus a summary
#                                          warning, rather than aborting the
#                                          whole batch)
#
# Resilience:
#   Each LLM call is attempted up to `max_attempts` times (default 3) with a
#   short exponential backoff between tries before it is counted as a failure.
#   Each attempt is also bounded by `timeout_secs` (default 120) via
#   setTimeLimit(), and the response is capped at `max_tokens = 4096` to keep
#   a doom-looping model from filling the context window. In the vector path,
#   documents that still fail return a row whose score columns are NA
#   (distinct from 0) and whose `length` and `elapsed_time` are still
#   populated. Structured responses that contain invalid field values also
#   preserve those fields as NA rather than converting them to zero.
#
# Runtime configuration:
#   PHQ9_LLM_PROVIDER  env var: "ollama" (default) or "openrouter".
#   PHQ9_LLM_MODEL     env var: model id; defaults to "llama3.2" for ollama
#                      and "anthropic/claude-haiku-4.5" for openrouter.
#   OPENROUTER_API_KEY env var: required when provider = "openrouter".
#   OLLAMA_BASE_URL    env var: ollama endpoint; defaults to localhost:11434.
#   `base_url` and `num_ctx` are Ollama-only and ignored for OpenRouter.
#   `workers` (score_documents_phq9_llm only): when > 1 and the `parallel`
#              package is available on a non-Windows platform, documents are
#              fanned out via parallel::mclapply(mc.preschedule = FALSE);
#              otherwise the call falls back to serial lapply.
#
# Expected runtime setup (Ollama path):
#   install.packages("ellmer")
#   ollama serve
#   ollama pull llama3.2
#
# Expected runtime setup (OpenRouter path):
#   install.packages("ellmer")
#   Sys.setenv(PHQ9_LLM_PROVIDER = "openrouter",
#              PHQ9_LLM_MODEL    = "anthropic/claude-haiku-4.5",
#              OPENROUTER_API_KEY = "sk-or-...")

if (!requireNamespace("ellmer", quietly = TRUE)) {
  stop(
    "CQH_PHQ9_LLM.R requires the 'ellmer' package. ",
    "Install with install.packages('ellmer')."
  )
}

PHQ9_LLM_FIELDS <- c(
  "phq9_total",
  "gad7_total",
  "irritability",
  "pain",
  "insomnia",
  "cognitive_complaints",
  "stressors",
  "depressed_mood",
  "anxiety"
)

PHQ9_LLM_FIELD_LIMITS <- c(
  phq9_total = 27,
  gad7_total = 21,
  irritability = 10,
  pain = 10,
  insomnia = 10,
  cognitive_complaints = 10,
  stressors = 10,
  depressed_mood = 10,
  anxiety = 10
)

PHQ9_LLM_OUTPUT_COLUMNS <- c(
  "length",
  PHQ9_LLM_FIELDS,
  "elapsed_time"
)

phq9_llm_system_prompt <- paste(
  "You are a skilled psychiatrist evaluating an outpatient primary care note.",
  "Ignore redacted materials.",
  "Estimate every requested score even if no formal test value is present.",
  "Return only numeric values in the requested JSON structure."
)

phq9_llm_user_prompt <- function(text) {
  paste0(
    "Based on the note, please report the following:\n",
    "1. Estimated PHQ-9 total score from 0-27,\n",
    "2. Estimated GAD-7 total score from 0-21,\n",
    "3. Extent of current irritability from 0-10; Score 0 if no symptoms are present and functioning is normal, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe,\n",
    "4. Extent of current pain from 0-10; Score 0 if no symptoms are present and functioning is normal, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe,\n",
    "5. Extent of current insomnia from 0-10; Score 0 if no symptoms are present and functioning is normal, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe,\n",
    "6. Extent of current cognitive complaints from 0-10; Score 0 if no symptoms are present and functioning is normal, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe,\n",
    "7. Extent of current stressors from 0-10; Score 0 if no stressors are present, 1-3 mild stressors, 4-6 moderate, 7-9 severe, 10 extremely severe,\n",
    "8. Extent of current depressed mood from 0-10; Score 0 if no symptoms are present, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe,\n",
    "9. Extent of current anxiety from 0-10; Score 0 if no symptoms are present, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe.\n\n",
    "The note is:\n",
    text,
    "\n\nReturn only a valid JSON string. You must return a number for each field. Estimate these scores even if no test value is present."
  )
}

phq9_llm_schema <- function() {
  ellmer::type_object(
    "Estimated PHQ-9, GAD-7, and related symptom burden scores for a single outpatient primary care note.",
    phq9_total = ellmer::type_number(
      "Estimated PHQ-9 total score, 0-27 inclusive."
    ),
    gad7_total = ellmer::type_number(
      "Estimated GAD-7 total score, 0-21 inclusive."
    ),
    irritability = ellmer::type_number(
      "Extent of current irritability, 0-10 inclusive."
    ),
    pain = ellmer::type_number(
      "Extent of current pain, 0-10 inclusive."
    ),
    insomnia = ellmer::type_number(
      "Extent of current insomnia, 0-10 inclusive."
    ),
    cognitive_complaints = ellmer::type_number(
      "Extent of current cognitive complaints, 0-10 inclusive."
    ),
    stressors = ellmer::type_number(
      "Extent of current stressors, 0-10 inclusive."
    ),
    depressed_mood = ellmer::type_number(
      "Extent of current depressed mood, 0-10 inclusive."
    ),
    anxiety = ellmer::type_number(
      "Extent of current anxiety, 0-10 inclusive."
    )
  )
}

phq9_normalize_note <- function(text) {
  if (!is.character(text) || length(text) != 1L || is.na(text) || !nzchar(text)) {
    return("")
  }

  text <- enc2utf8(text)
  text <- gsub("\r\n?", "\n", text, perl = TRUE)
  text <- gsub("[[:space:]]+", " ", text, perl = TRUE)
  trimws(text)
}

phq9_note_word_count <- function(text) {
  text <- phq9_normalize_note(text)
  if (!nzchar(text)) {
    return(0)
  }

  length(strsplit(text, "\\s+", perl = TRUE)[[1]])
}

empty_phq9_llm_score <- function(text = "", elapsed = 0) {
  vals <- c(phq9_note_word_count(text), rep(0, length(PHQ9_LLM_FIELDS)), elapsed)
  stats::setNames(vals, PHQ9_LLM_OUTPUT_COLUMNS)
}

failed_phq9_llm_score <- function(text = "", elapsed = 0) {
  vals <- c(
    phq9_note_word_count(text),
    rep(NA_real_, length(PHQ9_LLM_FIELDS)),
    elapsed
  )
  stats::setNames(vals, PHQ9_LLM_OUTPUT_COLUMNS)
}

phq9_validate_scalar_count <- function(x, arg_name) {
  x_int <- suppressWarnings(as.integer(x))
  if (!is.numeric(x) || length(x) != 1L || is.na(x) || !is.finite(x) ||
      is.na(x_int) || x < 1 || x != x_int) {
    stop("`", arg_name, "` must be a single integer >= 1.", call. = FALSE)
  }

  x_int
}

PHQ9_LLM_DEFAULT_MODELS <- c(
  ollama = "llama3.2",
  openrouter = "anthropic/claude-haiku-4.5"
)

resolve_phq9_model <- function(provider, model) {
  if (is.null(model) || !nzchar(model)) {
    model <- Sys.getenv("PHQ9_LLM_MODEL", "")
  }
  if (!nzchar(model)) {
    model <- PHQ9_LLM_DEFAULT_MODELS[[provider]]
  }
  if (is.null(model) || !nzchar(model)) {
    stop(
      "No PHQ/GAD LLM model resolved for provider '",
      provider,
      "'.",
      call. = FALSE
    )
  }
  model
}

make_phq9_chat <- function(provider = Sys.getenv("PHQ9_LLM_PROVIDER", "ollama"),
                           model = NULL,
                           base_url = Sys.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
                           system_prompt = phq9_llm_system_prompt,
                           params = ellmer::params(temperature = 0, seed = 20260411L, max_tokens = 4096L),
                           num_ctx = 32768L) {
  provider <- tolower(provider)
  if (!provider %in% names(PHQ9_LLM_DEFAULT_MODELS)) {
    stop(
      "Unknown PHQ/GAD LLM provider: '",
      provider,
      "'. Use 'ollama' or 'openrouter'.",
      call. = FALSE
    )
  }
  model <- resolve_phq9_model(provider, model)

  switch(provider,
    ollama = ellmer::chat_ollama(
      system_prompt = system_prompt,
      base_url = base_url,
      model = model,
      params = params,
      api_args = list(options = list(num_ctx = num_ctx))
    ),
    openrouter = ellmer::chat_openrouter(
      system_prompt = system_prompt,
      model = model,
      params = params
    )
  )
}

phq9_fresh_chat <- function(chat) {
  worker <- chat$clone()
  worker$set_turns(list())
  worker
}

parse_phq9_field_score <- function(field, x) {
  max_value <- PHQ9_LLM_FIELD_LIMITS[[field]]
  if (is.null(max_value)) {
    stop("Unknown PHQ/GAD score field: ", field, call. = FALSE)
  }
  if (!is.numeric(x) || length(x) != 1L || is.na(x) || !is.finite(x)) {
    return(NA_real_)
  }

  min(max_value, max(0, as.numeric(x)))
}

score_note_phq9_llm <- function(text, chat, schema = phq9_llm_schema(),
                                max_attempts = 3L, timeout_secs = 120L) {
  max_attempts <- phq9_validate_scalar_count(max_attempts, "max_attempts")
  timeout_secs <- phq9_validate_scalar_count(timeout_secs, "timeout_secs")
  last_err <- NULL

  for (attempt in seq_len(max_attempts)) {
    result <- tryCatch(
      {
        worker <- phq9_fresh_chat(chat)
        setTimeLimit(elapsed = timeout_secs, transient = TRUE)
        on.exit(setTimeLimit(), add = TRUE)
        out <- worker$chat_structured(phq9_llm_user_prompt(text), type = schema)
        setTimeLimit()
        out
      },
      error = function(e) {
        setTimeLimit()
        e
      }
    )

    if (!inherits(result, "error")) {
      return(vapply(
        PHQ9_LLM_FIELDS,
        function(field) parse_phq9_field_score(field, result[[field]]),
        numeric(1)
      ))
    }

    last_err <- result
    if (attempt < max_attempts) {
      Sys.sleep(0.5 * (2 ^ (attempt - 1L)))
    }
  }

  stop(conditionMessage(last_err), call. = FALSE)
}

#' Score a single clinical document with an ellmer-backed LLM.
#'
#' Returns a named numeric vector. `length` is note word count, the score
#' columns are burden estimates, and `elapsed_time` is wall-clock runtime in
#' seconds. When `chat` is supplied, `provider`, `model`, `base_url`, and
#' `num_ctx` are ignored. `base_url` and `num_ctx` only apply when
#' `provider = "ollama"`. `max_attempts` bounds the total number of tries
#' against the ellmer structured-output call; `timeout_secs` bounds the
#' wall-clock duration of each individual attempt via setTimeLimit(). This
#' function raises if every attempt fails. For partial-success semantics
#' across many documents, use `score_documents_phq9_llm()`.
score_document_phq9_llm <- function(text,
                                    chat = NULL,
                                    provider = Sys.getenv("PHQ9_LLM_PROVIDER", "ollama"),
                                    model = NULL,
                                    base_url = Sys.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
                                    num_ctx = 32768L,
                                    schema = phq9_llm_schema(),
                                    max_attempts = 3L,
                                    timeout_secs = 120L) {
  max_attempts <- phq9_validate_scalar_count(max_attempts, "max_attempts")
  timeout_secs <- phq9_validate_scalar_count(timeout_secs, "timeout_secs")

  if (!is.character(text) || length(text) != 1L) {
    stop(
      "score_document_phq9_llm() expects a length-1 character vector; ",
      "use score_documents_phq9_llm() for multiple documents."
    )
  }

  t0 <- Sys.time()
  note <- phq9_normalize_note(text)
  if (!nzchar(note)) {
    return(empty_phq9_llm_score(
      note,
      elapsed = as.numeric(difftime(Sys.time(), t0, units = "secs"))
    ))
  }

  total_length <- phq9_note_word_count(note)

  if (is.null(chat)) {
    chat <- make_phq9_chat(provider = provider, model = model,
                           base_url = base_url, num_ctx = num_ctx)
  }

  fields <- tryCatch(
    score_note_phq9_llm(note, chat = chat, schema = schema,
                        max_attempts = max_attempts,
                        timeout_secs = timeout_secs),
    error = function(e) {
      stop(
        "LLM scoring failed for the current document: ",
        conditionMessage(e),
        call. = FALSE
      )
    }
  )

  elapsed <- as.numeric(difftime(Sys.time(), t0, units = "secs"))

  stats::setNames(
    c(total_length, fields, elapsed),
    PHQ9_LLM_OUTPUT_COLUMNS
  )
}

try_score_document_phq9_llm <- function(text, chat, schema, max_attempts, timeout_secs) {
  t0 <- Sys.time()
  tryCatch(
    score_document_phq9_llm(text, chat = chat, schema = schema,
                            max_attempts = max_attempts,
                            timeout_secs = timeout_secs),
    error = function(e) {
      row <- failed_phq9_llm_score(
        if (is.character(text) && length(text) == 1L) text else "",
        elapsed = as.numeric(difftime(Sys.time(), t0, units = "secs"))
      )
      attr(row, "phq9_error") <- conditionMessage(e)
      row
    }
  )
}

# Append-only JSONL checkpoint for score_documents_phq9_llm().
#
# File layout: line 1 is a `_meta` JSON object pinning the run identity
# (texts hash + provider + model + field set + version). Subsequent lines are
# one-per-completed-document JSON objects with `index` plus the
# PHQ9_LLM_OUTPUT_COLUMNS fields. Only fully-valid rows
# (phq9_row_is_complete()) are appended; failures and partial-NA responses
# are absent so that a restart re-attempts them.

phq9_checkpoint_meta <- function(texts, provider, model) {
  list(
    version = 1L,
    hash = rlang::hash(texts),
    provider = provider,
    model = model,
    fields = PHQ9_LLM_FIELDS
  )
}

phq9_checkpoint_init <- function(path, meta) {
  line <- jsonlite::toJSON(list(`_meta` = meta), auto_unbox = TRUE)
  cat(line, "\n", sep = "", file = path, append = FALSE)
}

phq9_checkpoint_load <- function(path, expected_meta, n_texts) {
  lines <- readLines(path, warn = FALSE)
  if (length(lines) == 0L) {
    stop("Checkpoint file is empty: ", path, call. = FALSE)
  }

  header <- tryCatch(
    jsonlite::fromJSON(lines[[1L]], simplifyVector = FALSE),
    error = function(e) {
      stop("Checkpoint header is not valid JSON in ", path, ": ",
           conditionMessage(e), call. = FALSE)
    }
  )
  meta <- header$`_meta`
  if (is.null(meta)) {
    stop("Checkpoint header missing `_meta` in ", path, call. = FALSE)
  }
  for (key in c("version", "hash", "provider", "model", "fields")) {
    found <- unname(unlist(meta[[key]]))
    expect <- unname(unlist(expected_meta[[key]]))
    if (!identical(found, expect)) {
      stop("Checkpoint `", key, "` mismatch in ", path,
           ". Expected ", deparse(expect), " but found ", deparse(found),
           ". Use a different `checkpoint_path` or delete the file.",
           call. = FALSE)
    }
  }

  done <- list()
  for (i in seq_along(lines)[-1L]) {
    line <- lines[[i]]
    if (!nzchar(trimws(line))) next
    rec <- tryCatch(
      jsonlite::fromJSON(line, simplifyVector = TRUE),
      error = function(e) NULL
    )
    if (is.null(rec) || is.null(rec$index)) {
      warning("Skipping unparseable checkpoint line ", i, " in ", path,
              call. = FALSE)
      next
    }
    idx <- suppressWarnings(as.integer(rec$index))
    if (is.na(idx) || idx < 1L || idx > n_texts) {
      warning("Skipping checkpoint line ", i, " in ", path,
              " with out-of-range index ", rec$index, call. = FALSE)
      next
    }
    row <- vapply(PHQ9_LLM_OUTPUT_COLUMNS,
                  function(nm) {
                    v <- rec[[nm]]
                    if (is.null(v)) NA_real_ else as.numeric(v)
                  },
                  numeric(1))
    done[[as.character(idx)]] <- stats::setNames(row, PHQ9_LLM_OUTPUT_COLUMNS)
  }
  done
}

phq9_checkpoint_append <- function(path, index, row) {
  rec <- as.list(row)
  rec$index <- as.integer(index)
  line <- jsonlite::toJSON(rec, auto_unbox = TRUE, na = "null", digits = NA)
  cat(line, "\n", sep = "", file = path, append = TRUE)
}

phq9_row_is_complete <- function(row) {
  vals <- row[PHQ9_LLM_FIELDS]
  is.numeric(vals) && all(is.finite(vals))
}

#' Score many clinical documents with an ellmer-backed LLM.
#'
#' When `chat` is supplied, `provider`, `model`, `base_url`, and `num_ctx` are
#' ignored. `base_url` and `num_ctx` only apply when `provider = "ollama"`.
#' `workers > 1` fans the batch out via `parallel::mclapply` on POSIX systems
#' with the `parallel` package installed; otherwise the call runs serially.
#' `max_attempts` bounds the total number of tries per document and
#' `timeout_secs` bounds each attempt's wall-clock duration via
#' setTimeLimit(). A document that still fails after retries does not abort the
#' batch: its row has NA score columns and a single summary warning is emitted
#' listing the failed indices. Structured responses with invalid field values
#' keep those values as `NA`, and failed rows retain `elapsed_time`.
#'
#' `checkpoint_path` enables append-only JSONL durability. When set, every
#' fully-valid row is written to the file as it lands, keyed by absolute
#' index; restarting with the same path skips already-completed indices and
#' only re-attempts failures and missing rows. The checkpoint is pinned to
#' (`texts` hash, `provider`, `model`, field set) — any mismatch on restart
#' is a hard error. When `chat` is supplied alongside `checkpoint_path`, an
#' explicit `model` is required so the checkpoint records what produced the
#' scores.
score_documents_phq9_llm <- function(texts,
                                     chat = NULL,
                                     provider = Sys.getenv("PHQ9_LLM_PROVIDER", "ollama"),
                                     model = NULL,
                                     base_url = Sys.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
                                     num_ctx = 32768L,
                                     schema = phq9_llm_schema(),
                                     workers = 1L,
                                     max_attempts = 3L,
                                     timeout_secs = 120L,
                                     checkpoint_path = NULL) {
  workers <- phq9_validate_scalar_count(workers, "workers")
  max_attempts <- phq9_validate_scalar_count(max_attempts, "max_attempts")
  timeout_secs <- phq9_validate_scalar_count(timeout_secs, "timeout_secs")

  if (!is.character(texts)) {
    stop("score_documents_phq9_llm() expects a character vector.")
  }

  if (length(texts) == 0L) {
    return(as.data.frame(
      matrix(numeric(0), nrow = 0, ncol = length(PHQ9_LLM_OUTPUT_COLUMNS),
             dimnames = list(NULL, PHQ9_LLM_OUTPUT_COLUMNS))
    ))
  }

  chat_supplied <- !is.null(chat)

  done <- list()
  if (!is.null(checkpoint_path)) {
    if (chat_supplied && (is.null(model) || !nzchar(model))) {
      stop("`checkpoint_path` requires an explicit `model` argument when ",
           "`chat` is supplied; the model identity must be recorded in the ",
           "checkpoint meta.", call. = FALSE)
    }
    resolved_provider <- tolower(provider)
    if (!resolved_provider %in% names(PHQ9_LLM_DEFAULT_MODELS)) {
      stop("Unknown PHQ/GAD LLM provider: '", provider,
           "'. Use 'ollama' or 'openrouter'.", call. = FALSE)
    }
    resolved_model <- resolve_phq9_model(resolved_provider, model)
    meta <- phq9_checkpoint_meta(texts, resolved_provider, resolved_model)
    if (file.exists(checkpoint_path)) {
      done <- phq9_checkpoint_load(checkpoint_path, meta, length(texts))
    } else {
      phq9_checkpoint_init(checkpoint_path, meta)
    }
  }

  if (!chat_supplied) {
    chat <- make_phq9_chat(provider = provider, model = model,
                           base_url = base_url, num_ctx = num_ctx)
  }

  use_parallel <- workers > 1L &&
    .Platform$OS.type != "windows" &&
    requireNamespace("parallel", quietly = TRUE)

  todo_idx <- setdiff(seq_along(texts), as.integer(names(done)))

  if (!is.null(checkpoint_path)) {
    preview_n <- min(length(todo_idx), 5L)
    preview <- if (preview_n > 0L) {
      p <- paste(todo_idx[seq_len(preview_n)], collapse = ", ")
      if (length(todo_idx) > preview_n) paste0(p, ", ...") else p
    } else {
      "none"
    }
    message(sprintf(
      "checkpoint: %d/%d already complete, scoring %d this run (indices: %s)",
      length(done), length(texts), length(todo_idx), preview
    ))
  }

  score_one <- function(abs_i) {
    row <- try_score_document_phq9_llm(texts[[abs_i]], chat = chat,
                                       schema = schema,
                                       max_attempts = max_attempts,
                                       timeout_secs = timeout_secs)
    if (!is.null(checkpoint_path) && phq9_row_is_complete(row)) {
      phq9_checkpoint_append(checkpoint_path, abs_i, row)
    }
    row
  }

  batch_t0 <- Sys.time()
  new_rows <- if (use_parallel) {
    parallel::mclapply(todo_idx, score_one,
                       mc.cores = workers,
                       mc.preschedule = FALSE)
  } else {
    lapply(todo_idx, score_one)
  }

  batch_elapsed <- as.numeric(difftime(Sys.time(), batch_t0, units = "secs"))
  new_rows <- lapply(seq_along(new_rows), function(j) {
    r <- new_rows[[j]]
    if (inherits(r, "try-error")) {
      row <- failed_phq9_llm_score(texts[[todo_idx[[j]]]], elapsed = batch_elapsed)
      attr(row, "phq9_error") <- paste("worker error:",
                                       trimws(as.character(r)))
      return(row)
    }
    r
  })

  errors <- vapply(new_rows,
                   function(r) {
                     msg <- attr(r, "phq9_error")
                     if (is.null(msg)) NA_character_ else as.character(msg)
                   },
                   character(1))
  failed_local <- which(!is.na(errors))
  if (length(failed_local) > 0L) {
    failed_abs <- todo_idx[failed_local]
    preview_n <- min(length(failed_abs), 5L)
    preview <- paste(failed_abs[seq_len(preview_n)], collapse = ", ")
    if (length(failed_abs) > preview_n) {
      preview <- paste0(preview, ", ...")
    }
    warning(sprintf(
      "score_documents_phq9_llm(): %d of %d documents failed to score (indices: %s). Failed rows have NA score columns.",
      length(failed_abs), length(texts), preview
    ), call. = FALSE)
  }

  new_rows <- lapply(new_rows, function(r) { attr(r, "phq9_error") <- NULL; r })

  rows <- vector("list", length(texts))
  for (j in seq_along(todo_idx)) {
    rows[[todo_idx[[j]]]] <- new_rows[[j]]
  }
  for (k in names(done)) {
    rows[[as.integer(k)]] <- done[[k]]
  }

  if (!is.null(checkpoint_path)) {
    message(sprintf(
      "checkpoint: %d loaded, %d newly scored, %d failed",
      length(done),
      length(new_rows) - length(failed_local),
      length(failed_local)
    ))
  }

  as.data.frame(do.call(rbind, rows))
}

score_documents_phq9_llm

})


### Install Ollama

[Ollama](https://ollama.com/) is an open‑source runtime, [GUI](https://ollama.com/blog/new-app), and [API](https://docs.ollama.com/api/introduction) for running large language models locally. Local inference can be used for [real work](https://pubmed.ncbi.nlm.nih.gov/40350181/) on the [Apple silicon](https://developer.apple.com/videos/play/wwdc2020/10686/) chips many of you are [using right now](https://support.apple.com/en-us/116943).

The following commands are run on your colab VM to install ollama. If you want the GUI you can download it to your laptop from ollama.com and then run this code on your computer instead of Google's.

In [12]:
# Install ollama
system("curl -fsSL https://ollama.com/install.sh | sh")

# Is Ollama actually installed?
system("which ollama", intern = TRUE)

[1] "/usr/local/bin/ollama"

After you install ollama (whether on your computer or Google's) you need to start it. When you start it ollama will start a server on `localhost:11434` by default.

In [ ]:
# Start, give time to bind, confirm it is running.
system("bash -c 'nohup ollama serve > /content/ollama.log 2>&1 &'", wait = FALSE)
Sys.sleep(10)

#system("pgrep -a ollama", intern = TRUE)
#system("cat /content/ollama.log", intern = TRUE)
# Should return {"models":[]}
#system("curl -s http://localhost:11434/api/tags", intern = TRUE)
#system("ollama --version", intern = TRUE)
system("ss -tlnp 2>/dev/null | grep -E '11434|ollama'", intern = TRUE)

Once you have the Ollama runtime installed you will need to download an [open weight model](https://ollama.com/library?sort=popular) using `ollama pull <model-you-picked>`

We are going to use `llama3.2:3b` in the interest of speed.


In [ ]:
# Download model
system("ollama pull llama3.2:3b")

# Check that model was downloaded
#system("curl -s http://localhost:11434/api/tags", intern = TRUE)

In [ ]:
# @title
# Function for prompting ollama
ask_ollama <- function(prompt,
                       model = "llama3.2:3b",
                       system = "You are a helpful expert psychiatrist.",
                       host = "http://localhost:11434") {

  httr2::request(host) |>
  httr2::req_url_path("/api/generate") |>
  httr2::req_body_json(list(model = model,
                              system = system,
                              prompt = prompt,
                              stream = FALSE,
                              options = list(temperature = 0,
                                             seed = 42))) |>
  httr2::req_timeout(300) |>
  httr2::req_perform() |>
  httr2::resp_body_json() |>
  _$response
}

In [ ]:
system("gunzip -k sobp_nlp.csv.gz")

# Start Here

Everything up to this point was setting up the colab virtual machine to support this work. Moving forward you will be running code against that virtual machine. You could be running this code against your laptop, a VM in your data center, or elsewhere.

Some of this code will produce network requests. Those network requests can either target this machine - in which case the fact of the network request is merely an implementation detail - or they could target other machines - in which case you are crossing a trust boundary.

Here we have no confidentiality concerns because our data and code are all public. In any real use you will have confidentiality concerns and thus crossing trust boundaries is of critical concern. Do not use this code with real patient data without a clear understanding of the privacy implications.

## Our data

For this demo we will use a sample from the [medical transcripts public data](https://www.kaggle.com/datasets/tboyle10/medicaltranscriptions/data)

In [ ]:
d <- read.csv("sobp_nlp.csv", stringsAsFactors = F)
message("This is what we've got")
str(head(d))

## RDoC Bag of Words

Start with RDoC phenotypes from a bag of words, published in PMID [29496195](https://pubmed.ncbi.nlm.nih.gov/29496195/)


The Bag of Words is just what it sounds like in that the note to be scored becomes an unordered list of words in which we search for a selected group of magic words. The work is in selecting the magic words. Here, for example, are the magic words for the negative domain:

```r
negative = c(
    "aggression", "aggressive", "agitated", "anger", "angry",
    "anxiety", "anxiety_disorder", "anxious", "appetite", "ashamed",
    "assaultive", "attack", "avoidance", "avoidant", "blue",
    "calm", "chest", "chest_pain", "chill", "constricted",
    "depressed", "depressed_mood", "depressive", "depressive_disorder", "diaphoretic",
    "diarrhea", "difficulty", "down", "dysphoric", "dysthymic",
    "emotional", "energy", "fatigue", "fearful", "feelings",
    "fever", "frighten", "gain", "guilt", "guilty",
    "head", "helpless", "hopeful", "hopelessness", "hypervigilant",
    "irritable", "lesion", "living", "major", "meaningless",
    "nausea", "nervous", "optimistic", "outburst", "palpitation",
    "panic", "panic_attack", "panic_disorder", "pessimistic", "phobia",
    "phobic", "posttraumatic", "posttraumatic_stress", "refuse", "relaxed",
    "respond", "sad", "scare", "sensory", "sensory_deficit",
    "situation", "startle", "stress", "stress_disorder", "stressor",
    "suicidal", "sweat", "tachycardic", "tearful", "tense",
    "threaten", "trauma", "tremulous", "violence", "violent",
    "weight_gain", "worried", "worth", "worthless"
  )
```

All of the concepts and lexical variations are in the source code above where `score_rdoc_bow` is defined.


In [ ]:
message("Score a note from our data set")
score_rdoc_bow(d$note[1])

Notice that you have 5 domains, not 6, when this was developed RDoC had 5 domains and the time it takes to score is very short.

It's so short you could just go ahead and score all the notes.

In [ ]:
bow <- score_rdoc_bow(d$note)
str(bow)

In [ ]:
hist(log(bow$elapsed_time), main = "Time to score RDoC BoW, log(sec)")
message("It was quick: ", sum(bow$elapsed_time))

## RDoC SQL

Because this version of the RDoC score runs on token matching alone it is easy to port to different runtimes and environments.

Many organizations use SQL to distribute clinical data for secondary use. With more complex models you would need to extract the text (creating a copy with all the associated governance risk) for scoring. With a token matching approach you can avoid making a copy or even exposing the source data to investigators by running the score within SQL.

As a demo consider the [Sqlite](https://sqlite.org/) schema stored in `RDOC_SQL_SCHEMA` as an example:


In [ ]:
cat(substr(RDOC_SQL_SCHEMA, 1, 500))
print('...')

To use this we need to do roughly the following:

```bash

$ sqlite3 rdoc.db < RDOC_SQL_SCHEMA.sql

$ sqlite3 rdoc.db <<'EOF'
.mode csv
.import sobp_nlp.csv notes_staging
INSERT INTO notes(id, note_text)
  SELECT nid, note FROM notes_staging;
EOF

$ sqlite3 -header -csv rdoc.db "SELECT * FROM cqh_note_scores;" > sql_scores.csv

```


In [ ]:
# Setup the sqlite database
system2("sqlite3", args = "rdoc.db", input = RDOC_SQL_SCHEMA)

# Load our sample data into the database
system("sqlite3 rdoc.db <<'EOF'
.mode csv
.import sobp_nlp.csv notes_staging
INSERT INTO notes(id, note_text)
  SELECT nid, note FROM notes_staging;
EOF")

# Use the database to score (this will take a little while)
system("sqlite3 -header -csv rdoc.db 'SELECT * FROM cqh_note_scores;' > sql_scores.csv")

Now that you've run your scores in Sql load the scores up into R so we can look at them

In [ ]:
bow_sql <- read.csv("sql_scores.csv")
str(bow_sql)

Lets look at how similar the Sql and the R version of the score are per domain:

In [ ]:
rbind("negative" = cor.test(bow_sql$negative, bow$negative)$est,
      "positive" = cor.test(bow_sql$positive, bow$positive)$est,
      "social" = cor.test(bow_sql$social, bow$social)$est,
      "cognitive" = cor.test(bow_sql$cognitive, bow$cognitive)$est,
      "arousal_regulatory" = cor.test(bow_sql$arousal_regulatory, bow$arousal_regulatory)$est
      )

They should be slightly different given some of the runtime differences in how string cleaning works.

## RDoC LLM

### Using the LLM

Now lets use an LLM. For this we are going to get to the LLM by a network request; however, the request will be a network request from this VM to this VM and thus we will not cross a trust boundary.

You could run this on your laptop - I've done so for [clinical prediction](https://mentalhealth.bmj.com/content/28/1/e301654) in the past.

This is where we set up the request

```r
httr2::req_body_json(list(model = "llama3.2:3b",
                          system = "You are a helpful expert in psychiatry.",
                          prompt = "What is SOBP?",
                          stream = FALSE,
                          options = list(
                            temperature = 0,
                            seed = 42))
```


Lets put together a request that we will send over the network (to ourselves)

In [ ]:
httr2::request("http://localhost:11434") |>
httr2::req_url_path("/api/generate") |>
httr2::req_body_json(list(model = "llama3.2:3b",
                          system = "You are a helpful expert in psychiatry.",
                          prompt = "What is SOBP?",
                          stream = FALSE,
                          options = list(
                            temperature = 0,
                            seed = 42))
                          ) |>
    httr2::req_timeout(300) |>
    httr2::req_dry_run()

Now run the request for real and get the response:

In [ ]:
httr2::request("http://localhost:11434") |>
httr2::req_url_path("/api/generate") |>
httr2::req_body_json(list(model = "llama3.2:3b",
                          system = "You are a helpful expert in psychiatry.",
                          prompt = "What is SOBP?",
                          stream = FALSE,
                          options = list(
                            temperature = 0,
                            seed = 42))
                          ) |>
httr2::req_timeout(300) |>
httr2::req_perform() |>
httr2::resp_body_json() -> llm_rsp


IRdisplay::display_json(llm_rsp)

We can wrap all of that code up in a function (`ask_ollama`) to make this easier to reuse.

In [ ]:
ask_ollama("What is SOBP?")

We are using a very small model. It will run on a CPU but it's going to be very slow. If you get a free T4 VM or have a paid account such that you can be sure to get a GPU this should run very quickly. Although it is fast it's going to be slow vs BoW.

In [ ]:
message("Our request took: ", llm_rsp$total_duration / 1e9, " seconds")

### Using the RDoC LLM

RDoC by LLM is just a model, a prompt, and a note. This was published in: McCoy TH, Perlis RH. Dimensional Measures of Psychopathology in Children and Adolescents Using Large Language Models. *Biol Psychiatry*. 2024;96(12):940-947. doi: 10.1016/j.biopsych.2024.05.008. PMID: [38866172](https://pubmed.ncbi.nlm.nih.gov/38866172/) and this is the prompt:


> You are a skilled psychiatrist scoring an emergency room clinical note in terms of how the patient symptoms over the past 24 hours reflect the 6 NIMH Research Domain Criteria (RDoC): Negative Valence Systems, Positive Valence Systems, Cognitive Systems, Social Processes, Arousal and Regulatory Systems, and Sensorimotor Systems.
>
> Remember that substance use can be reflected as Positive Valence symptom.
>
> Notes are scored on a 0-10 scale to capture the magnitude of documented symptoms relevant in a given domain. Score 0 if no symptoms are present and functioning is normal, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe. Generally moderate requires treatment and severe requires hospitalization.
>
> Return only a valid JSON object with each score, Negative, Positive, Cognitive, Social, Arousal, and Sensorimotor.


Use of this is all wrapped up in `score_rdoc_llm`

#### Start with local LLM

In [ ]:
message("This is going to take a bit, even though we are only scoring 2 notes")
llm <- score_rdoc_llm(d$note[1:2])
str(llm)

In [ ]:
message("These two RDoC scores by LLM took ",
        sum(llm$elapsed_time),
        " whereas 3000 by BOW took ",
        sum(bow$elapsed_time))

The time delta here speaks to the difference in work between the token matching and LLM approach. LLMs are energy intensive. Management of this energy demand is among the [costs / harms](https://news.harvard.edu/gazette/story/2026/04/why-are-communities-pushing-back-against-data-centers/) that must be managed when using an LLM.

In the interest of time I have pre cooked the results and you can load those pre-cooked results here:


In [ ]:
llm <- readRDS('llama.rds')
str(llm)

message("Total time ", sum(llm$elapsed_time), " seconds")

This is a mixed bag.


1. We now have a `sensorimotor` domain and we got it ~ "for free."
2. We ran on an LLM (with all the reading comprehension you see routinely) without crossing a trust boundary
3. We ran on my laptop in a plausible time

But the scores look very different


In [ ]:
rbind("negative" = cor.test(llm$negative, bow$negative)$est,
      "positive" = cor.test(llm$positive, bow$positive)$est,
      "social" = cor.test(llm$social, bow$social)$est,
      "cognitive" = cor.test(llm$cognitive, bow$cognitive)$est,
      "arousal_regulatory" = cor.test(llm$arousal, bow$arousal_regulatory)$est
      )

#### Now try a remote LLM

What if we cross a trust boundary and ask Claude instead?

To do this you need to set some environment variables to change the behavior of the various LLM score functions

```r
Sys.setenv(RDOC_LLM_PROVIDER = "openrouter",
          RDOC_LLM_MODEL    = "anthropic/claude-sonnet-4.6",
          OPENROUTER_API_KEY = "... you gotta have one of these ...")
```

Load up pre baked results from Sonnet 4.6 (remember don't do this with real clinical data).


In [ ]:
rmt_llm <- readRDS('sonnet46.rds')
str(rmt_llm)

message("Total time ", sum(rmt_llm$elapsed_time), " seconds")

In [ ]:
rbind("negative" = cor.test(rmt_llm$negative, bow$negative)$est,
      "positive" = cor.test(rmt_llm$positive, bow$positive)$est,
      "social" = cor.test(rmt_llm$social, bow$social)$est,
      "cognitive" = cor.test(rmt_llm$cognitive, bow$cognitive)$est,
      "arousal_regulatory" = cor.test(rmt_llm$arousal, bow$arousal_regulatory)$est
      )

In [ ]:
reset_rdoc_ollama_env()

#### What do these models know?

##### Local Llama

What does the local model know about RDoC?

```r
cat(ask_ollama("Tell me about each of the NIMH RDoC domains"))
```

> The National Institute of Mental Health's Research Domain Criteria (RDoC) is a framework for organizing and understanding mental health conditions based on their underlying neurobiological mechanisms, rather than relying solely on symptom-based diagnoses. The RDoC framework identifies five core domains:

> 1. **Positive Valence System**: This domain encompasses experiences related to pleasure, reward, and positive emotions. It includes symptoms such as:
	* Increased interest in activities
	* Enhanced enjoyment of previously pleasurable activities
	* Feelings of happiness or euphoria

> Examples of conditions associated with this domain include:
	* Bipolar disorder (mania/hypomania)
	* Substance use disorders (e.g., cocaine, amphetamines)

> 2. **Negative Valence System**: This domain involves experiences related to pain, distress, and negative emotions. It includes symptoms such as:
	* Increased anxiety or worry
	* Feelings of sadness, hopelessness, or despair
	* Difficulty sleeping

> Examples of conditions associated with this domain include:
	* Major depressive disorder (MDD)
	* Anxiety disorders (e.g., generalized anxiety disorder, social anxiety disorder)

> 3. **Reward System**: This domain is related to the brain's reward circuitry and involves experiences that provide a sense of pleasure or satisfaction. It includes symptoms such as:
	* Increased motivation or drive
	* Enhanced focus or concentration
	* Feelings of accomplishment or pride

> Examples of conditions associated with this domain include:
	* Attention Deficit Hyperactivity Disorder (ADHD)
	* Obsessive-compulsive disorder (OCD)

> 4. **Stress System**: This domain involves experiences related to stress, arousal, and the body's response to threats. It includes symptoms such as:
	* Increased heart rate or blood pressure
	* Difficulty relaxing or calming down
	* Hypervigilance

> Examples of conditions associated with this domain include:
	* Post-traumatic stress disorder (PTSD)
	* Generalized anxiety disorder

> 5. **Cognitive System**: This domain encompasses experiences related to attention, perception, and cognitive processing. It includes symptoms such as:
	* Difficulty concentrating or paying attention
	* Distorted perceptions or hallucinations
	* Disorganized thinking or speech

> Examples of conditions associated with this domain include:
	* Schizophrenia
	* Traumatic brain injury (TBI)

> It's essential to note that these domains are not mutually exclusive, and many mental health conditions can be characterized by symptoms across multiple domains. The RDoC framework aims to provide a more nuanced understanding of mental health conditions and their underlying neurobiological mechanisms.

> Keep in mind that the RDoC is still an evolving framework, and research continues to refine our understanding of these domains and their relationships with various mental health conditions.


##### Remote Claude

What does Sonnet 4.6 know?


> `Tell me about each of the NIMH RDoC domains`


> The NIMH Research Domain Criteria (RDoC) framework organizes mental function into **6 domains**, each reflecting distinct neurobehavioral systems. Here's a concise overview of each:

> **1. Negative Valence Systems**
Governs responses to aversive situations — fear, threat, loss, and frustration. Key constructs include acute threat ("fear"), potential threat ("anxiety"), sustained threat, loss, and frustrative non-reward. This domain maps most closely onto mood and anxiety disorders but is broadly relevant across psychopathology.

> **2. Positive Valence Systems**
Covers motivational and reward processes — approach behavior, reward anticipation, consummatory pleasure, reward learning, and habit formation. Distinctions between *wanting* (anticipation/motivation) and *liking* (hedonic response) are central here. Anhedonia and addiction both implicate this domain.

> **3. Cognitive Systems**
Encompasses higher-order cognitive processes: attention, perception, declarative memory, language behavior, cognitive control, and working memory. Relevant to a wide range of conditions including schizophrenia, ADHD, PTSD, and dementia-related syndromes.

> **4. Social Processes**
Addresses how organisms perceive and interact with others. Constructs include affiliation and attachment, social communication, perception and understanding of self, and perception and understanding of others (including Theory of Mind). Central to autism spectrum disorder research but broadly applicable.

> **5. Arousal and Regulatory Systems**
Covers arousal, circadian rhythms, and sleep-wake regulation. These systems modulate responsiveness to stimuli and coordinate internal state across behavioral domains. Disruptions cut across virtually all psychiatric conditions.

> **6. Sensorimotor Systems**
The most recently added domain (formalized around 2019). Covers motor action, sensorimotor dynamics, and the interaction between sensory input and motor output — including inhibition/facilitation of action, agency, and sensorimotor integration. Relevant to tic disorders, catatonia, psychosis, and developmental conditions.

> **A few framing notes worth keeping in mind:**
The RDoC framework is explicitly *dimensional* rather than categorical — these domains are meant to be studied across the full range from normal to abnormal, using multiple *units of analysis* (genes, molecules, cells, circuits, physiology, behavior, self-report). The constructs within each domain are meant to be relatively independent neurobehavioral units rather than symptom clusters, which distinguishes RDoC from DSM-style taxonomy. In practice, most psychiatric conditions implicate multiple domains simultaneously.

Does it know so much that this is a little bit circular?

If we ask it about what it knows from the various publications using the BoW model has the model (without access to memory or web grounding) baked in some knowledge of the prior work?

> `What can you tell me about the use of NLP to extract RDoC symptom burdens from EHRs? TH McCoy and RH Perlis from MGH / HMS have done work in this area.`

> This is a rich intersection of clinical informatics, psychiatry, and computational linguistics. Here's what I can tell you about this area, including the work from McCoy and Perlis:
>
> ## RDoC and EHRs: The Core Challenge
>
> The **Research Domain Criteria (RDoC)** framework, developed by NIMH, aims to classify mental illness based on dimensions of neurobiology and behavior rather than traditional DSM categories. Extracting RDoC-relevant constructs from EHRs is appealing because:
>
> - EHRs contain rich, longitudinal clinical narratives in free text
> - Structured data alone (ICD codes, labs) captures only a fraction of symptom burden
> - RDoC dimensions like **negative valence, positive valence, cognitive systems, arousal/regulatory systems**, and **social processes** are often described descriptively by clinicians
>
> ## NLP Approaches in This Space
>
> **General methods used:**
>
> - **Rule-based / lexicon approaches** — keyword matching for symptom terms, with negation detection (e.g., NegEx)
> - **Supervised classification** — training models on annotated clinical notes to detect RDoC constructs
> - **Transformer-based models** — BERT variants (BioBERT, ClinicalBERT) fine-tuned on clinical text for concept extraction
> - **Weak supervision / distant supervision** — using structured codes or scales as labels for training without full manual annotation
>
> ## McCoy & Perlis Contributions (MGH/HMS)
>
> Their group has been particularly productive in this area. Key themes in their work include:
>
> **Symptom burden extraction from notes:**
> - Using NLP to extract psychiatric symptom dimensions from clinical notes at scale, enabling phenotyping across large EHR cohorts (Partners/Mass General Brigham)
> - Mapping extracted constructs to RDoC domains, going beyond DSM diagnoses
>
> **Specific constructs targeted:**
>- **Suicidality and self-harm** — NLP to identify suicidal ideation documented in notes, including passive ideation that often doesn't trigger structured codes
>- **Anhedonia and positive valence deficits** — hard to capture in ICD codes but frequently mentioned in notes
>- **Anxiety, fear, and threat responsiveness** — negative valence constructs
>- **Sleep and arousal disturbance**
>
>**Methodological work:**
>- Validating NLP-extracted phenotypes against gold-standard chart review
>- Demonstrating that NLP-derived symptom burden adds predictive value over diagnosis codes alone (e.g., for outcomes like hospitalization, suicide attempt)
>- Using these phenotypes in **pharmacogenomic** and **treatment response** studies — associating symptom profiles with medication outcomes
>
>**Scale:**
>- Their work leverages the Mass General Brigham EHR system, giving access to millions of patient records, which enables well-powered association studies
>
>## Why This Matters Clinically and Scientifically
>
>- **Transdiagnostic research** — RDoC dimensions cut across diagnoses; NLP-extracted features enable studying e.g. anhedonia regardless of whether the patient carries MDD, bipolar, or schizophrenia
>- **Reducing ascertainment bias** — structured codes undercount symptom burden, especially in patients with less severe presentations or those seen in primary care
>- **Enabling precision psychiatry** — linking symptom-level phenotypes to genetics, biomarkers, or treatment response
>
>## Limitations and Open Problems
>
>- **Annotation bottleneck** — creating ground-truth labels for training is expensive and requires clinical expertise
>- **Negation and speculation** — clinical notes are full of "denies SI," "no evidence of," "rule out" — handling this correctly is critical
>- **Temporal context** — distinguishing current vs. historical symptoms
>- **Generalizability** — models trained at one health system often don't transfer well to others due to note style differences
>- **Implicit expression** — clinicians often *imply* symptom burden without stating it explicitly
>
>If you want, I can go deeper on any particular aspect — the NLP methodology, the RDoC construct mapping, the specific McCoy/Perlis papers, or the downstream applications in pharmacogenomics and outcomes research?



### How about an LLM for PHQ9

This was published in:

McCoy TH, Castro VM, Perlis RH. Estimating depression severity in narrative clinical notes using large language models. *J Affect Disord*. 2025;381:270-274. doi: 10.1016/j.jad.2025.04.014. PMID: [40187432](https://pubmed.ncbi.nlm.nih.gov/40187432/).

and is wrapped up as `score_phq9_llm`

Which uses a system prompt

> You are a skilled psychiatrist evaluating an outpatient primary care note. Ignore redacted materials. Estimate every requested score even if no formal test value is present. Return only numeric values in the requested JSON structure.

and a templated message:

```
    "Based on the note, please report the following:\n",
    "1. Estimated PHQ-9 total score from 0-27,\n",
    "2. Estimated GAD-7 total score from 0-21,\n",
    "3. Extent of current irritability from 0-10; Score 0 if no symptoms are present and functioning is normal, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe,\n",
    "4. Extent of current pain from 0-10; Score 0 if no symptoms are present and functioning is normal, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe,\n",
    "5. Extent of current insomnia from 0-10; Score 0 if no symptoms are present and functioning is normal, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe,\n",
    "6. Extent of current cognitive complaints from 0-10; Score 0 if no symptoms are present and functioning is normal, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe,\n",
    "7. Extent of current stressors from 0-10; Score 0 if no stressors are present, 1-3 mild stressors, 4-6 moderate, 7-9 severe, 10 extremely severe,\n",
    "8. Extent of current depressed mood from 0-10; Score 0 if no symptoms are present, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe,\n",
    "9. Extent of current anxiety from 0-10; Score 0 if no symptoms are present, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe.\n\n",
    "The note is:\n",
    text,
    "\n\nReturn only a valid JSON string. You must return a number for each field. Estimate these scores even if no test value is present."
  ```

Score one note locally on ollama model. It's a complex prompt so it is going to run for a too long for colab unless you are on a GPU.

In [ ]:
phq <- score_phq9_llm(d$note[1])

Load pre baked PHQ9 results run on GPT5. As before this move to a remote LLM means crossing a trust boundary.

In [ ]:
phq <- readRDS('gpt5_phq9.rds')
str(phq)

message("Total time ", sum(phq$elapsed_time), " seconds")

### RDoC vs PHQ9

Lets look at how RDoC and PHQ9 relate to one another.

In [ ]:
rbind("negative" = cor.test(rmt_llm$negative, phq$phq9_total)$est,
      "positive" = cor.test(rmt_llm$positive, phq$phq9_total)$est,
      "social" = cor.test(rmt_llm$social, phq$phq9_total)$est,
      "cognitive" = cor.test(rmt_llm$cognitive,phq$phq9_total)$est,
      "arousal_regulatory" = cor.test(rmt_llm$arousal, phq$phq9_total)$est,
      "motor" = cor.test(rmt_llm$sensorimotor, phq$phq9_total)$est
      )

## Structured output

The `"type": "json_schema"` [structured response](https://ellmer.tidyverse.org/articles/structured-data.html) which can be [enforced](https://platform.claude.com/docs/en/build-with-claude/structured-outputs) by [models](https://developers.openai.com/api/docs/guides/structured-outputs) is a game changer.


Without structured output, you ask for JSON and hope the model obeys:

> "Please return JSON only"

But the model might return:

>Sure, here is the JSON:
>{ ... }

or omit a field, rename a field, include commentary, use "mild" instead of 2, or produce malformed JSON.

Your code then needs fragile parsing, retries, regex cleanup, and lots of edge-case handling.

With structured output, the schema is part of the API request:
```json
"response_format": {
  "type": "json_schema",
  ...
}
```

That changes the contract. The model is not merely prompted to produce the right shape; the serving layer constrains or validates the response so it matches the schema.

#### Define in code

You can then use this to define the schema and validate to local types.


```r
ellmer::type_object(
    "Estimated PHQ-9, GAD-7, and related symptom burden scores for a single outpatient primary care note.",
    phq9_total = ellmer::type_number(
      "Estimated PHQ-9 total score, 0-27 inclusive."
    ),
    gad7_total = ellmer::type_number(
      "Estimated GAD-7 total score, 0-21 inclusive."
    ),
    irritability = ellmer::type_number(
      "Extent of current irritability, 0-10 inclusive."
    ),
    pain = ellmer::type_number(
      "Extent of current pain, 0-10 inclusive."
    ),
    insomnia = ellmer::type_number(
      "Extent of current insomnia, 0-10 inclusive."
    ),
    cognitive_complaints = ellmer::type_number(
      "Extent of current cognitive complaints, 0-10 inclusive."
    ),
    stressors = ellmer::type_number(
      "Extent of current stressors, 0-10 inclusive."
    ),
    depressed_mood = ellmer::type_number(
      "Extent of current depressed mood, 0-10 inclusive."
    ),
    anxiety = ellmer::type_number(
      "Extent of current anxiety, 0-10 inclusive."
    )
)
```

#### which will get you json

Which will produce the following request in which *"Return only numeric values in the requested JSON structure."* is no longer required.


```json
{
  "messages": [
    {
      "role": "system",
      "content": "You are a skilled psychiatrist evaluating an outpatient primary care note. Ignore redacted materials. Estimate every requested score even if no formal test value is present. Return only numeric values in the requested JSON structure."
    },
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "Based on the note, please report the following:\n1. Estimated PHQ-9 total score from 0-27,\n2. Estimated GAD-7 total score from 0-21,\n3. Extent of current irritability from 0-10; Score 0 if no symptoms are present and functioning is normal, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe,\n4. Extent of current pain from 0-10; Score 0 if no symptoms are present and functioning is normal, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe,\n5. Extent of current insomnia from 0-10; Score 0 if no symptoms are present and functioning is normal, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe,\n6. Extent of current cognitive complaints from 0-10; Score 0 if no symptoms are present and functioning is normal, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe,\n7. Extent of current stressors from 0-10; Score 0 if no stressors are present, 1-3 mild stressors, 4-6 moderate, 7-9 severe, 10 extremely severe,\n8. Extent of current depressed mood from 0-10; Score 0 if no symptoms are present, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe,\n9. Extent of current anxiety from 0-10; Score 0 if no symptoms are present, 1-3 mild symptoms, 4-6 moderate, 7-9 severe, 10 extremely severe.\n\nThe note is:\nPREOPERATIVE DIAGNOSIS: , Benign prostatic hypertrophy.,POSTOPERATIVE DIAGNOSIS: , Benign prostatic hypertrophy.,SURGERY: ,Cystopyelogram and laser vaporization of the prostate.,ANESTHESIA: , Spinal.,ESTIMATED BLOOD LOSS: , Minimal ... <truncated>
        }
      ]
    }
  ],
  "model": "openai/gpt-5-nano",
  "temperature": 0,
  "seed": 20260411,
  "max_tokens": 4096,
  "stream": false,
  "response_format": {
    "type": "json_schema",
    "json_schema": {
      "name": "structured_data",
      "schema": {
        "type": "object",
        "description": "Estimated PHQ-9, GAD-7, and related symptom burden scores for a single outpatient primary care note.",
        "properties": {
          "phq9_total": {
            "type": "number",
            "description": "Estimated PHQ-9 total score, 0-27 inclusive."
          },
          "gad7_total": {
            "type": "number",
            "description": "Estimated GAD-7 total score, 0-21 inclusive."
          },
          "irritability": {
            "type": "number",
            "description": "Extent of current irritability, 0-10 inclusive."
          },
          "pain": {
            "type": "number",
            "description": "Extent of current pain, 0-10 inclusive."
          },
          "insomnia": {
            "type": "number",
            "description": "Extent of current insomnia, 0-10 inclusive."
          },
          "cognitive_complaints": {
            "type": "number",
            "description": "Extent of current cognitive complaints, 0-10 inclusive."
          },
          "stressors": {
            "type": "number",
            "description": "Extent of current stressors, 0-10 inclusive."
          },
          "depressed_mood": {
            "type": "number",
            "description": "Extent of current depressed mood, 0-10 inclusive."
          },
          "anxiety": {
            "type": "number",
            "description": "Extent of current anxiety, 0-10 inclusive."
          }
        },
        "required": [
          "phq9_total",
          "gad7_total",
          "irritability",
          "pain",
          "insomnia",
          "cognitive_complaints",
          "stressors",
          "depressed_mood",
          "anxiety"
        ],
        "additionalProperties": false
      },
      "strict": true
    }
  }
}
```
